# UPAR 2027 Track 1 — notebook e2e cho bản 0.8017

Notebook này dựng lại bản nộp `SUBMIT_split_x2f_cmask_f8cL2c_tail4flip_gpu_w8.zip` (bản gốc md5 `e9bb5cdf`, test **0.8017**) trong **một notebook**, đi từ dữ liệu của BTC (annotation task1 + ảnh Market1501 / PA100k / PETA) tới file zip nộp. Mọi mã train / suy luận / đóng gói nằm trong các cell; notebook không import file `.py` nào của project và không ghi file `.py` nào ra đĩa để import.

**`run.py` và `model.py`**: định dạng nộp của BTC bắt buộc zip có `run.py` (chương trình chấm import nó rồi gọi `predict_batch`) và `model.py`. Hai file này được viết thẳng trong 2 cell `%%module run` / `%%module model`. Magic `%%module` (cell 1) chỉ lưu mã nguồn của cell và nạp nó thành module trong bộ nhớ (`exec`), rồi khi đóng gói thì ghi nguyên văn bản đó vào zip. Byte của 2 file trong zip trùng bản gốc (notebook kiểm CRC).

**Lưu ý: bản 0.8017 KHÔNG sạch.** Cách phủ đuôi (ảnh Synth, vị trí ≥ 28095: thêm g11 + RADIO, lật ngang, trọng số 0.24/0.16/0.3/0.3) được chọn sau khi xem điểm test của nhiều bản nộp. Notebook chỉ tái tạo đúng công thức của bản đó.

## Thành phần

| weights | model | train | ảnh train | nhãn | dùng ở |
|---|---|---|---|---|---|
| `weights_0` | DINOv3 ViT-H+/16 (`dinov3_vith16p`), đóng băng 8 khối đầu | `e2e_f8c` | 256x128 | CMASK | đầu + đuôi (w 0.24) |
| `weights_1` | DINOv2 ViT-L/14 reg (`dinov2_vitl14_reg`) | `e2e_l2c` | 252x126 | CMASK | đầu + đuôi (w 0.16) |
| `weights_2` | DINOv3 ViT-H+/16, đóng băng 20 khối đầu | `e2e_g11` | 256x128 | gốc | chỉ đuôi (w 0.3) |
| `weights_3` | C-RADIOv4-H (`radio_v4h`), đóng băng 12 khối đầu | `e2e_radio` | 256x128 | gốc | chỉ đuôi (w 0.3) |

- Đầu (MEVID) chạy ở 224x112; đuôi (Synth) ở tail_tier 1: 192x96 cho DINOv3/RADIO, 196x98 cho DINOv2. Nén int8 theo kênh (w8).
- **Đầu** (MEVID, vị trí < 28095): 0.6·F8c + 0.4·L2c, ngưỡng `thresholds_solo`.
- **Đuôi** (Synth): 4 model + lật ngang, ngưỡng `thresholds`.
- **CMASK**: 2 model DINOv3-L (`e2e_oof_f0/f1`, chia train theo nhóm thành 2 fold) dự đoán chéo phần train còn lại. Ô nhãn nào bị cả biên logit (|z − logit(ngưỡng)| > 2) lẫn xác suất Platt (≥ 0.9 / ≤ 0.1) phản đối thì bị bỏ khỏi loss (mask = 0). Nhãn không đổi.
- Train / val: train + val chính thức gộp lại rồi chia lại (15 % mỗi dataset làm val, chia theo nhóm, iterative stratification, seed 0). Ngưỡng hiệu chỉnh trên val này.

## Chạy theo stage (nhiều phiên)

Toàn bộ quy trình mất khoảng 30 giờ GPU, quá 12 giờ của một phiên Kaggle. `UPAR_STAGES` (biến môi trường, hoặc sửa ở cell cấu hình) chọn stage chạy trong phiên. Kết quả mỗi lượt train nằm ở `/kaggle/working/e2e_runs/<tên>/`. Phiên sau gắn Output của phiên trước làm **Input**, notebook tự tìm và dùng lại.

| phiên | phần cứng | `UPAR_STAGES` | ước lượng | cần trước |
|---|---|---|---|---|
| A | T4 x2 | `oof0,oof1,flags` | ~6 h | — |
| B | T4 x2 | `l2c` | ~5 h | A (flags) |
| C | T4 x2 | `g11` | ~7 h | — |
| D | T4 x2 | `radio` | ~6.5 h | — |
| E | GPU ≥ 40 GB (A100) | `f8c` | | A (flags) |
| F | GPU bất kỳ | `pack` | ~30 phút | B, C, D, E |

- Một lượt train chạy quá `UPAR_TIME_LIMIT_H` (mặc định 11 h) thì tự lưu `resume.pt` trước epoch kế tiếp và dừng. Chạy lại cùng stage (gắn Output cũ làm Input) sẽ train tiếp.
- `f8c` cần GPU ≥ 39 GB (A100 / Colab), T4 không đủ bộ nhớ. `UPAR_FORCE_F8C=1` bỏ qua kiểm tra này.

**Không trùng byte với bản gốc:** train trên GPU không tất định, nên weights và ngưỡng sẽ khác bản gốc. Điểm test dự kiến gần 0.80, không đảm bảo đúng 0.8017. Các phần tất định thì notebook kiểm bằng md5 / CRC: chia train/val, `run.py`, `model.py`, `metadata.yaml`, cấu trúc `config.json`.

Biến môi trường khác:
- `UPAR_DATA_DIR`: thư mục ảnh.
- `UPAR_INPUT_ROOT`: gốc Input, mặc định `/kaggle/input`.
- `UPAR_WORKERS`: số worker DataLoader.
- Kaggle Secret `WACV-UPAR-NTN` (token Hugging Face): không bắt buộc. Thiếu token thì DINOv3-L lấy bản timm công khai, cùng weights.

In [ ]:
print("[cell] setup", flush=True)
import os, sys, io, re, gc, csv, json, time, math, zlib, types, random, shutil, hashlib, zipfile, linecache, subprocess, urllib.request
from pathlib import Path
import numpy as np, pandas as pd
from IPython.core.magic import register_cell_magic

T_START = time.time()
IS_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "work"
INPUT_ROOT = Path(os.environ.get("UPAR_INPUT_ROOT", "/kaggle/input"))
RUNS = WORK / "e2e_runs"          # mỗi lượt train một thư mục con
PKG = WORK / "package"            # assets/ của gói nộp (config.json, weights_*.pt)
for p in (WORK, RUNS, PKG / "assets"): p.mkdir(parents=True, exist_ok=True)

SRC = {}   # mã nguồn các cell %%module: "model" (= model.py của gói nộp), "run" (= run.py)

def load_source(name, path=None):
    """Nạp SRC[name] thành module `name` trong bộ nhớ (sys.modules). Không đọc file nào: `path` chỉ đặt __file__
    (run.py tìm assets/ cạnh __file__)."""
    fn = str(path or PKG / f"{name}.py"); src = SRC[name]
    mod = types.ModuleType(name); mod.__file__ = fn
    sys.modules[name] = mod
    linecache.cache[fn] = (len(src), None, src.splitlines(True), fn)      # traceback chỉ đúng dòng trong cell
    exec(compile(src, fn, "exec", dont_inherit=True), mod.__dict__)
    return mod

@register_cell_magic
def module(line, cell):
    """%%module NAME: lưu nguyên văn cell vào SRC[NAME] rồi nạp thành module NAME (dòng log [cell] in ở đây)."""
    name = line.strip()
    print(f"[cell] module {name}", flush=True)
    SRC[name] = cell
    load_source(name)
    print(f"  module {name}: {len(cell.splitlines())} dòng, đã nạp", flush=True)

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh ngoài, đẩy stdout/stderr ra log từng dòng."""
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for ln in p.stdout: print("  " + ln.rstrip(), flush=True)
    rc = p.wait()
    if check and rc: raise RuntimeError(f"lệnh lỗi (exit {rc}): {cmd}")
    return rc

def seed_all(s):
    random.seed(s); np.random.seed(s)
    import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)

_IDX = {}
def input_dirs(root=None, max_depth=10, max_files=300):
    # Chỉ mục thư mục trong Input, duyệt 1 lần. Không đi vào thư mục có > max_files file (thư mục ảnh).
    root = str(root or INPUT_ROOT)
    if root not in _IDX:
        t0, out, stack = time.time(), [], [(Path(root), 0)]
        while stack:
            d, k = stack.pop(); subs, nf = [], 0
            try:
                with os.scandir(d) as it:
                    for e in it:
                        if e.is_dir(follow_symlinks=True): subs.append(e.path)
                        else:
                            nf += 1
                            if nf > max_files: break
            except OSError:
                continue
            if nf > max_files: continue
            out.append(d)
            if k < max_depth: stack += [(Path(x), k + 1) for x in subs]
        _IDX[root] = out
        print(f"Chỉ mục Input {root}: {len(out)} thư mục ({time.time() - t0:.0f}s)", flush=True)
    return _IDX[root]

print("WORK =", WORK, "| Input =", INPUT_ROOT, "| Kaggle" if IS_KAGGLE else "| ngoài Kaggle")

## Cấu hình

`STAGES` chọn việc của phiên này (xem bảng ở đầu). Lượt train đã có `meta.json` (trong `/kaggle/working/e2e_runs` hoặc Input) thì được dùng lại, không train lại.

In [ ]:
print("[cell] config", flush=True)
ALL_STAGES = ["oof0", "oof1", "flags", "l2c", "g11", "radio", "f8c", "pack"]
STAGES = [s for s in os.environ.get("UPAR_STAGES", ",".join(ALL_STAGES)).split(",") if s]
assert set(STAGES) <= set(ALL_STAGES), f"stage lạ: {set(STAGES) - set(ALL_STAGES)}"
TIME_LIMIT_H = float(os.environ.get("UPAR_TIME_LIMIT_H", "11"))   # quá mốc này (tính từ đầu notebook) thì lưu resume.pt và dừng
SEED = 42
WORKERS = int(os.environ.get("UPAR_WORKERS", min(4, os.cpu_count() or 1)))
SPLIT_AT = 28095                                                   # test: MEVID #0-28094, Synth #28095-29247
ZIP_NAME = "SUBMIT_split_x2f_cmask_f8cL2c_tail4flip_gpu_w8.zip"
print("STAGES =", STAGES, "| TIME_LIMIT_H =", TIME_LIMIT_H, "| WORKERS =", WORKERS)

In [ ]:
print("[cell] hf-token", flush=True)
# Token Hugging Face chỉ để tải repo gated facebook/dinov3-vitl16 (model OOF). Không có token -> bản timm công khai (cùng weights).
HF_SECRET = "WACV-UPAR-NTN"
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret(HF_SECRET)
    except Exception:
        pass
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get(HF_SECRET) or ""
    except Exception:
        pass
print("HF_TOKEN:", "có" if os.environ.get("HF_TOKEN") else "không có (DINOv3-L lấy từ timm)")

## 1. Dữ liệu BTC

- **Annotation task1** và `peta_file_mapping.txt`: tìm trong Input. Không có thì `git clone` repo BTC `speckean/UPAR-Challenge-2027`.
- **Ảnh**: dataset nào đã có trong Input (đúng cấu trúc `Market1501/…`, `PA100k/…`, `PETA/images/…`) thì symlink sang. Thiếu dataset nào thì tải bằng link BTC, theo đúng các bước của `download_datasets.py` (viết lại ở đây, không import file của BTC):
  - Market1501: Google Drive.
  - PA100k: Google Drive.
  - PETA: Dropbox, rồi đổi tên ảnh theo `peta_file_mapping.txt`.

In [ ]:
print("[cell] data", flush=True)
BTC_REPO = "https://github.com/speckean/UPAR-Challenge-2027"
MARKET_URL = "https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ"
PA100K_ID = "1Bod3MrbdCRiSvpR7NSOYi-rF3-ushwtj"            # PA100k/data.zip trong thư mục Drive của BTC
PA100K_FOLDER = "https://drive.google.com/drive/folders/1d_D0Yh7C262gr0ef9EqkvG_M3fqgAWa2?usp=sharing"
PETA_URL = "https://www.dropbox.com/s/52ylx522hwbdxz6/PETA.zip?dl=1"
PROBE = {"Market1501": "Market1501/bounding_box_train/0002_c1s1_000451_03.jpg",
         "PA100k":     "PA100k/release_data/release_data/000001.jpg",
         "PETA":       "PETA/images/00001.png"}

def find_btc_repo():
    for d in input_dirs():
        if (d / "data/annotations/task1/train/gt.csv").exists() and (d / "peta_file_mapping.txt").exists():
            return d
    repo = WORK / "UPAR-Challenge-2027"
    if not (repo / "data/annotations/task1/train/gt.csv").exists():
        shutil.rmtree(repo, ignore_errors=True)
        sh(["git", "clone", "-q", "--depth", "1", BTC_REPO, repo])
    return repo

def extract_zip(z, dst):
    # như extract_zip của BTC: bỏ qua mục lỗi thay vì dừng
    with zipfile.ZipFile(z) as zf:
        items = zf.infolist(); print(f"  giải nén {z.name}: {len(items)} mục", flush=True)
        for k, m in enumerate(items):
            try: zf.extract(m, dst)
            except zipfile.error as e: print(f"  bỏ qua {m.filename}: {e}", flush=True)
            if k and k % 20000 == 0: print(f"  ... {k}/{len(items)}", flush=True)

def gdown_mod():
    try:
        import gdown
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", "gdown==5.2.0"])   # gdown >= 6 bỏ tham số fuzzy
        import gdown
    return gdown

def http_download(url, dst):
    print(f"  tải {url} -> {dst}", flush=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(dst, "wb") as f:
        n, nxt = 0, 0
        while (b := r.read(1 << 22)):
            f.write(b); n += len(b)
            if n >= nxt: print(f"  ... {n / 2**20:.0f} MB", flush=True); nxt += 200 * 2**20

def get_market(dst):
    gdown = gdown_mod(); z = dst / "market_1501.zip"
    gdown.download(MARKET_URL, output=str(z), quiet=False, use_cookies=False, fuzzy=True)
    extract_zip(z, dst); (dst / "Market-1501-v15.09.15").rename(dst / "Market1501"); z.unlink()

def get_pa100k(dst):
    gdown = gdown_mod(); p = dst / "PA100k"; p.mkdir(parents=True, exist_ok=True); z = p / "data.zip"
    ok = lambda: z.exists() and z.stat().st_size > 400 * 2**20
    try: gdown.download(id=PA100K_ID, output=str(z), quiet=False)
    except Exception as e: print("  gdown theo id lỗi:", type(e).__name__, flush=True)
    if not ok(): gdown.download_folder(PA100K_FOLDER, output=str(p), quiet=False, use_cookies=False)   # cách của BTC
    assert ok(), "không tải được PA100k/data.zip (Google Drive hết lượt tải?) -> gắn PA100k vào Input rồi chạy lại"
    extract_zip(z, p); z.unlink()

def get_peta(dst, mapping_file):
    p = dst / "PETA"; p.mkdir(parents=True, exist_ok=True); z = p / "peta.zip"
    http_download(PETA_URL, z); extract_zip(z, p); z.unlink(); (p / "images").mkdir(exist_ok=True)
    with open(mapping_file, newline="", encoding="utf-8") as f: mapping = {r[0]: r[1] for r in csv.reader(f) if r}
    n = miss = 0
    for fp in p.glob("*/*/*/*"):                                   # PETA/PETA dataset/<bộ con>/archive/<ảnh>
        if fp.suffix == ".txt" or not fp.is_file(): continue
        key = fp.relative_to(dst).as_posix()
        if key not in mapping: miss += 1; continue
        shutil.move(str(fp), str(dst / mapping[key])); n += 1
    print(f"  PETA: {n} ảnh -> PETA/images ({miss} file không có trong mapping)", flush=True)

REPO = find_btc_repo(); ANN_BTC = REPO / "data/annotations/task1"
print("annotation BTC:", ANN_BTC)
DATA = Path(os.environ.get("UPAR_DATA_DIR") or ("/tmp/upar_data" if IS_KAGGLE else WORK / "data"))
DATA.mkdir(parents=True, exist_ok=True)
GETTERS = {"Market1501": get_market, "PA100k": get_pa100k, "PETA": lambda d: get_peta(d, REPO / "peta_file_mapping.txt")}
for name, rel in PROBE.items():
    if (DATA / rel).exists(): print(f"[{name}] đã có trong {DATA}"); continue
    src = next((d for d in input_dirs() if (d / rel).exists()), None)
    for q in (DATA / name, DATA / "Market-1501-v15.09.15"):
        if q.is_symlink() or q.is_file(): q.unlink()
        elif q.exists() and (name == "Market1501" or q.name == name): shutil.rmtree(q)
    if src is not None:
        os.symlink(src / name, DATA / name); print(f"[{name}] symlink -> {src / name}"); continue
    print(f"[{name}] không có trong Input -> tải bằng link BTC", flush=True)
    GETTERS[name](DATA)
    assert (DATA / rel).exists(), f"[{name}] tải xong nhưng thiếu {rel}"
    print(f"[{name}] OK", flush=True)
print("DATA =", DATA)

## 2. Chia lại train / val (resplit 0710)

- Gộp train + val chính thức của từng dataset, lấy 15 % làm val mới. Chia theo nhóm:
  - Market: mã người.
  - PA100k: khối 20 chỉ số.
  - PETA: khối 5 chỉ số.
- Cân bằng nhãn bằng iterative stratification (Sechidis 2011) ở mức nhóm, seed 0.
- Kết quả tất định. Notebook kiểm md5 so với bản đã dùng để train bản gốc.

In [ ]:
print("[cell] resplit", flush=True)
RESPLIT = WORK / "resplit_0710/annotations/task1"
RESPLIT_MD5 = {"train": "cfdd75df57949fe066ca5d60a0b6375f", "val": "957280e4591e2997d659adb65ae97d3f"}   # md5 sau khi đổi CRLF -> LF

def strat_groups(Y, sizes, f, rng):
    """Y: (nhóm, thuộc tính) số ảnh dương; sizes: số ảnh mỗi nhóm. Trả về mảng bool: nhóm vào val."""
    ratios = np.array([f, 1 - f]); need = ratios[:, None] * Y.sum(0)[None, :]; need_n = ratios * sizes.sum()
    fold = np.full(len(sizes), -1); left = np.ones(len(sizes), bool)
    while left.any():
        c = Y[left].sum(0)
        if not (c > 0).any():                                   # nhóm không còn nhãn dương nào: chia theo số ảnh còn thiếu
            for g in rng.permutation(np.where(left)[0]):
                k = int(np.argmax(need_n)); fold[g] = k; need_n[k] -= sizes[g]; left[g] = False
            break
        j = int(np.argmin(np.where(c > 0, c, np.inf)))          # thuộc tính hiếm nhất còn lại
        for g in rng.permutation(np.where(left & (Y[:, j] > 0))[0]):
            cand = np.flatnonzero(need[:, j] == need[:, j].max())
            if len(cand) > 1: cand = cand[need_n[cand] == need_n[cand].max()]
            k = int(rng.choice(cand)); fold[g] = k; need[k] -= Y[g]; need_n[k] -= sizes[g]; left[g] = False
    return fold == 0

def make_resplit(ann, out, f=0.15, seed=0):
    raw = {s: pd.read_csv(ann / s / "gt.csv") for s in ("train", "val")}
    col0 = raw["train"].columns[0]; attr = list(raw["train"].columns[1:]); assert list(raw["val"].columns) == list(raw["train"].columns)
    d = pd.concat([raw["train"].assign(old="train"), raw["val"].assign(old="val")], ignore_index=True)
    d["ds"] = d[col0].str.split("/").str[0]
    num = d[col0].str.extract(r"(\d+)\.(?:jpg|png)$")[0].astype(float).fillna(-1)   # 19 ảnh Market đuôi ".jpg.jpg" (Market chia theo mã người)
    pid = d[col0].str.extract(r"/(-?\d+)_c\d")[0]
    d["grp"] = np.where(d.ds == "Market1501", "M" + pid.fillna("x"),
               np.where(d.ds == "PA100k", "A" + (num // 20).astype(int).astype(str), "P" + (num // 5).astype(int).astype(str)))
    rng = np.random.default_rng(seed); d["new"] = "train"
    for ds, x in d.groupby("ds"):
        g = x.groupby("grp"); Y = g[attr].sum().values.astype(float); sizes = g.size().values.astype(float)
        to_val = strat_groups(Y, sizes, f, rng)
        d.loc[x.index[x.grp.isin(g.size().index[to_val])], "new"] = "val"
    for s in ("train", "val"):
        p = out / s; p.mkdir(parents=True, exist_ok=True)
        d[d.new == s][[col0] + attr].to_csv(p / "gt.csv", index=False, lineterminator="\n")
    return d

sp_ = make_resplit(ANN_BTC, RESPLIT)
print(sp_.groupby(["ds", "new"]).size().unstack().to_string())
for s in ("train", "val"):
    h = hashlib.md5((RESPLIT / s / "gt.csv").read_bytes().replace(b"\r\n", b"\n")).hexdigest()
    print(f"resplit {s}: md5 {h} ({'TRÙNG' if h == RESPLIT_MD5[s] else 'KHÁC'} bản gốc)")
    assert h == RESPLIT_MD5[s] or os.environ.get("UPAR_ALLOW_SPLIT_MISMATCH"), "chia train/val khác bản gốc (annotation BTC đổi?)"

In [ ]:
print("[cell] frames", flush=True)
DS_NAMES = ["Market1501", "PA100k", "PETA"]
DS_ID = {d: i for i, d in enumerate(DS_NAMES)}

def load(split):
    d = pd.read_csv(RESPLIT / split / "gt.csv"); d = d.rename(columns={d.columns[0]: "image"}); d["split"] = split
    return d
df = pd.concat([load("train"), load("val")], ignore_index=True)
ATTR = [c for c in df.columns if c not in ("image", "split")]
assert len(ATTR) == 40
assert all(a.startswith("UpperBody-Color") for a in ATTR[8:20]) and all(a.startswith("LowerBody-Color") for a in ATTR[21:33])
df["ds"] = df.image.str.split("/").str[0]

# MASK (dataset, thuộc tính) = 0: dataset không gán nhãn thuộc tính đó -> bỏ khỏi loss
MASK = np.ones((len(DS_NAMES), 40), np.float32)
for d, g in df[df.split == "train"].groupby("ds"):
    MASK[DS_ID[d], g[ATTR].values.sum(0) == 0] = 0
for a in ["UpperBody-Color-Other", "LowerBody-Color-Other"]:
    MASK[DS_ID["PETA"], ATTR.index(a)] = 0

TR = df[df.split == "train"].reset_index(drop=True); VA = df[df.split == "val"].reset_index(drop=True)
assert len(VA) == 19780, len(VA)

def pos_rate(frame):
    m = MASK[frame.ds.map(DS_ID).values]
    return ((frame[ATTR].values * m).sum(0) / np.maximum(m.sum(0), 1)).clip(1e-4, 1 - 1e-4).astype(np.float32)

rng_ = np.random.RandomState(0)
miss = [p for p in df.image.values[rng_.choice(len(df), 3000, replace=False)] if not (DATA / p).exists()]
assert not miss, f"thiếu ảnh, ví dụ {miss[:3]}"
print("train:", TR.ds.value_counts().to_dict(), "| val:", VA.ds.value_counts().to_dict(), "| kiểm 3000 ảnh ngẫu nhiên: có đủ")

## 3. `model.py` của gói nộp

Nguyên văn `model.py` trong zip bản gốc (60 822 byte). Định nghĩa `PARNet` và các backbone (DINOv2, DINOv3, DINOv3 ViT-H+, C-RADIOv4…) bằng PyTorch thuần (server chấm không có mạng, không có timm). Trong file có cả các nhánh không dùng ở bản này (CLIP, SigLIP, PE-Core, đầu truy vấn…), giữ nguyên để byte trùng bản gốc. Cell `%%module model` nạp mã này thành module `model` trong bộ nhớ.

In [ ]:
%%module model

import math, torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torch.utils.checkpoint
import torchvision.transforms.functional as TF

IMAGENET_NORM = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
CLIP_NORM = ((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
MEAN, STD = IMAGENET_NORM            # giữ tương thích với bản cũ

SIGLIP_NORM = ((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

def norm_for(arch):
    return CLIP_NORM if arch.startswith(("clip", "radio")) else SIGLIP_NORM if arch.startswith(("siglip", "pe_core")) else IMAGENET_NORM

def grayworld(t):
    # Cân bằng trắng gray-world trên tensor [0,1] (3,H,W): đưa trung bình 3 kênh về bằng nhau, giới hạn hệ số 0.6-1.6
    m = t.mean((1, 2)); g = (m.mean() / m.clamp(min=1e-3)).clamp(0.6, 1.6)
    return (t * g[:, None, None]).clamp(0, 1)

def crop_frac(img, crop):
    # 09/10: cắt theo tỷ lệ (x0, y0, x1, y1) của ảnh gốc, trước mọi phép resize (model chuyên vùng đầu). crop rỗng -> giữ nguyên ảnh
    if not crop: return img
    W, H = img.size; x0, y0, x1, y1 = crop
    l, t = int(round(x0 * W)), int(round(y0 * H))
    return img.crop((l, t, max(int(round(x1 * W)), l + 1), max(int(round(y1 * H)), t + 1)))

def letterbox(img, h, w, mean=IMAGENET_NORM[0]):
    # 11/10 (E31 lbox): giữ tỷ lệ khung — co ảnh vừa khít (w, h) rồi đệm 2 bên / trên dưới bằng màu trung bình (= 0 sau chuẩn hoá)
    s = min(w / img.width, h / img.height); nw, nh = max(1, round(img.width * s)), max(1, round(img.height * s))
    from PIL import Image as _I
    c = _I.new("RGB", (w, h), tuple(int(round(v * 255)) for v in mean)); c.paste(img.resize((nw, nh), resample=2), ((w - nw) // 2, (h - nh) // 2))
    return c

def preprocess(img, h, w, mean=IMAGENET_NORM[0], std=IMAGENET_NORM[1], wb=False, lbox=False):
    # PIL RGB -> tensor chuẩn hoá. Dùng chung cho val (notebook) và inference (run.py).
    img = letterbox(img, h, w, mean) if lbox else img.resize((w, h), resample=2)          # 2 = BILINEAR
    t = TF.to_tensor(img)
    if wb: t = grayworld(t)
    return TF.normalize(t, mean, std)

def _drop_path(x, p, training):
    if p == 0.0 or not training: return x
    keep = 1 - p
    return x * x.new_empty(x.shape[0], 1, 1).bernoulli_(keep) / keep

STRIPES = ((0.0, 0.30), (0.20, 0.60), (0.50, 1.00))      # tỷ lệ chiều cao: đầu, thân trên, thân dưới

def _stripe_means(tok, grid):
    # tok: (B, gh*gw, C) patch token theo thứ tự hàng -> trung bình từng dải ngang
    B, N, C = tok.shape; gh, gw = grid
    t = tok.reshape(B, gh, gw, C)
    return [t[:, int(round(a * gh)):max(int(round(a * gh)) + 1, int(round(b * gh)))].mean((1, 2)) for a, b in STRIPES]

def _resize_pos(grid_w, new_grid, antialias=True):
    # grid_w: (N, C) lưới vuông -> nội suy bicubic sang new_grid (h, w)
    s = int(math.sqrt(grid_w.shape[0])); C = grid_w.shape[1]
    g = grid_w.reshape(1, s, s, C).permute(0, 3, 1, 2).float()
    g = F.interpolate(g, size=new_grid, mode="bicubic", align_corners=False, antialias=antialias)
    return g.permute(0, 2, 3, 1).reshape(-1, C), s

# ------------------------------ DINOv2 ViT (key khớp checkpoint chính thức của Meta) ------------------------------
class _Attn(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); self.h = heads
        self.qkv = nn.Linear(dim, dim * 3); self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(B, N, C))

class _Mlp(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__(); self.fc1 = nn.Linear(dim, hidden); self.act = nn.GELU(); self.fc2 = nn.Linear(hidden, dim)
    def forward(self, x): return self.fc2(self.act(self.fc1(x)))

class _LayerScale(nn.Module):
    def __init__(self, dim): super().__init__(); self.gamma = nn.Parameter(torch.ones(dim) * 1e-5)
    def forward(self, x): return x * self.gamma

class _Block(nn.Module):
    def __init__(self, dim, heads, dp):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6); self.attn = _Attn(dim, heads); self.ls1 = _LayerScale(dim)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6); self.mlp = _Mlp(dim, dim * 4); self.ls2 = _LayerScale(dim)
        self.dp = dp
    def forward(self, x):
        x = x + _drop_path(self.ls1(self.attn(self.norm1(x))), self.dp, self.training)
        return x + _drop_path(self.ls2(self.mlp(self.norm2(x))), self.dp, self.training)

class _PatchEmbed(nn.Module):
    def __init__(self, dim, patch): super().__init__(); self.proj = nn.Conv2d(3, dim, patch, patch)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class DinoViT(nn.Module):
    def __init__(self, dim, depth, heads, grid, n_reg=0, patch=14, drop_path=0.0, parts=False):
        super().__init__()
        self.grid, self.n_reg, self.parts = grid, n_reg, parts
        self.patch_embed = _PatchEmbed(dim, patch)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + grid[0] * grid[1], dim))
        self.register_tokens = nn.Parameter(torch.zeros(1, n_reg, dim)) if n_reg else None
        dps = [drop_path * i / max(1, depth - 1) for i in range(depth)]
        self.blocks = nn.ModuleList([_Block(dim, heads, d) for d in dps])
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.out_dim = (2 + 3 * parts) * dim
    def features(self, x, taps=()):
        # (đặc trưng gộp, patch token cuối, patch token sau các khối trong taps)
        B = x.shape[0]
        x = torch.cat([self.cls_token.expand(B, -1, -1), self.patch_embed(x)], 1) + self.pos_embed
        if self.n_reg:
            x = torch.cat([x[:, :1], self.register_tokens.expand(B, -1, -1), x[:, 1:]], 1)
        mids = []
        for i, b in enumerate(self.blocks):
            x = b(x)
            if i in taps: mids.append(x[:, 1 + self.n_reg:])
        x = self.norm(x); pt = x[:, 1 + self.n_reg:]
        f = torch.cat([x[:, 0], pt.mean(1)] + (_stripe_means(pt, self.grid) if self.parts else []), 1)   # CLS + trung bình patch (+ 3 dải ngang)
        return f, pt, mids
    def forward(self, x, return_tokens=False):
        f, pt, _ = self.features(x)
        return (f, pt) if return_tokens else f

DINO_CFG = {"dinov2_vits14": (384, 12, 6), "dinov2_vitb14": (768, 12, 12), "dinov2_vitl14": (1024, 24, 16)}

def load_dinov2_pretrained(vit, name, n_reg):
    url = f"https://dl.fbaipublicfiles.com/dinov2/{name}/{name}{'_reg4' if n_reg else ''}_pretrain.pth"
    sd = torch.hub.load_state_dict_from_url(url, map_location="cpu")
    pe = sd["pos_embed"]
    grid, s = _resize_pos(pe[0, 1:], vit.grid)
    sd["pos_embed"] = torch.cat([pe[:, :1], grid[None].to(pe.dtype)], 1)
    sd.pop("mask_token", None)
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, (missing, unexpected)
    print(f"Loaded {url} | pos_embed {s}x{s} -> {vit.grid[0]}x{vit.grid[1]}")

# ------------------------------ CLIP ViT (key khớp HuggingFace CLIPVisionModel + visual_projection) ------------------------------
class _QuickGELU(nn.Module):
    def forward(self, x): return x * torch.sigmoid(1.702 * x)

class _ClipAttn(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); self.h = heads
        self.q_proj = nn.Linear(dim, dim); self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim); self.out_proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape; h = self.h
        q = self.q_proj(x).view(B, N, h, C // h).transpose(1, 2)
        k = self.k_proj(x).view(B, N, h, C // h).transpose(1, 2)
        v = self.v_proj(x).view(B, N, h, C // h).transpose(1, 2)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.out_proj(x.transpose(1, 2).reshape(B, N, C))

class _ClipMlp(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__(); self.fc1 = nn.Linear(dim, hidden); self.act = _QuickGELU(); self.fc2 = nn.Linear(hidden, dim)
    def forward(self, x): return self.fc2(self.act(self.fc1(x)))

class _ClipLayer(nn.Module):
    def __init__(self, dim, heads, dp):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(dim, eps=1e-5); self.self_attn = _ClipAttn(dim, heads)
        self.layer_norm2 = nn.LayerNorm(dim, eps=1e-5); self.mlp = _ClipMlp(dim, dim * 4)
        self.dp = dp
    def forward(self, x):
        x = x + _drop_path(self.self_attn(self.layer_norm1(x)), self.dp, self.training)
        return x + _drop_path(self.mlp(self.layer_norm2(x)), self.dp, self.training)

class _ClipEmb(nn.Module):
    def __init__(self, dim, patch, grid):
        super().__init__()
        self.class_embedding = nn.Parameter(torch.zeros(dim))
        self.patch_embedding = nn.Conv2d(3, dim, patch, patch, bias=False)
        self.position_embedding = nn.Embedding(1 + grid[0] * grid[1], dim)

class _ClipEncoder(nn.Module):
    def __init__(self, dim, depth, heads, dps):
        super().__init__(); self.layers = nn.ModuleList([_ClipLayer(dim, heads, d) for d in dps])

class ClipViT(nn.Module):
    def __init__(self, dim, depth, heads, grid, patch, proj_dim, drop_path=0.0, parts=False):
        super().__init__()
        self.grid, self.dim, self.proj_dim, self.parts = grid, dim, proj_dim, parts
        self.embeddings = _ClipEmb(dim, patch, grid)
        self.pre_layrnorm = nn.LayerNorm(dim, eps=1e-5)          # tên giống hệt HF (kể cả lỗi chính tả)
        dps = [drop_path * i / max(1, depth - 1) for i in range(depth)]
        self.encoder = _ClipEncoder(dim, depth, heads, dps)
        self.post_layernorm = nn.LayerNorm(dim, eps=1e-5)
        self.visual_projection = nn.Linear(dim, proj_dim, bias=False)
        self.out_dim = proj_dim + (1 + 3 * parts) * dim
    @property
    def blocks(self): return self.encoder.layers
    def embed(self, x):
        B = x.shape[0]
        p = self.embeddings.patch_embedding(x).flatten(2).transpose(1, 2)
        c = self.embeddings.class_embedding.to(p.dtype).expand(B, 1, -1)
        x = torch.cat([c, p], 1) + self.embeddings.position_embedding.weight[None].to(p.dtype)
        x = self.pre_layrnorm(x)
        for l in self.encoder.layers: x = l(x)
        return x
    def image_embeds(self, x):                                  # giống CLIPModel.get_image_features
        return self.visual_projection(self.post_layernorm(self.embed(x)[:, 0]))
    def forward(self, x, return_tokens=False):
        x = self.embed(x)
        e = F.normalize(self.visual_projection(self.post_layernorm(x[:, 0])).float(), dim=-1)   # embedding ảnh CLIP
        f = [e, self.post_layernorm(x[:, 1:].mean(1)).float()]                                  # trung bình patch
        if self.parts: f += [self.post_layernorm(s).float() for s in _stripe_means(x[:, 1:], self.grid)]
        f = torch.cat(f, 1)
        return (f, self.post_layernorm(x[:, 1:])) if return_tokens else f

CLIP_CFG = {"clip_vitb16": ("openai/clip-vit-base-patch16", 768, 12, 12, 16, 512),
            "clip_vitl14": ("openai/clip-vit-large-patch14", 1024, 24, 16, 14, 768)}

def clip_state_from_hf(hf_sd, grid):
    sd = {k[len("vision_model."):]: v for k, v in hf_sd.items() if k.startswith("vision_model.")}
    sd["visual_projection.weight"] = hf_sd["visual_projection.weight"]
    sd.pop("embeddings.position_ids", None)
    pe = sd["embeddings.position_embedding.weight"]
    g, s = _resize_pos(pe[1:], grid)
    sd["embeddings.position_embedding.weight"] = torch.cat([pe[:1].float(), g], 0).to(pe.dtype)
    return sd, s

def load_clip_pretrained(vit, arch):
    from transformers import CLIPModel            # chỉ cần khi train, run.py không dùng
    hf = CLIPModel.from_pretrained(CLIP_CFG[arch][0]).state_dict()
    sd, s = clip_state_from_hf(hf, vit.grid)
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, (missing, unexpected)
    print(f"Loaded {CLIP_CFG[arch][0]} | pos_embed {s}x{s} -> {vit.grid[0]}x{vit.grid[1]}")

# ------------------------------ SigLIP / SigLIP 2 ViT (key khớp HuggingFace SiglipVisionModel) ------------------------------
class _GELUTanh(nn.Module):
    def forward(self, x): return F.gelu(x, approximate="tanh")

class _SigMlp(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__(); self.fc1 = nn.Linear(dim, hidden); self.act = _GELUTanh(); self.fc2 = nn.Linear(hidden, dim)
    def forward(self, x): return self.fc2(self.act(self.fc1(x)))

class _SigLayer(nn.Module):
    def __init__(self, dim, heads, dp, hidden=None):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(dim, eps=1e-6); self.self_attn = _ClipAttn(dim, heads)
        self.layer_norm2 = nn.LayerNorm(dim, eps=1e-6); self.mlp = _SigMlp(dim, hidden or dim * 4)
        self.dp = dp
    def forward(self, x):
        x = x + _drop_path(self.self_attn(self.layer_norm1(x)), self.dp, self.training)
        return x + _drop_path(self.mlp(self.layer_norm2(x)), self.dp, self.training)

class _SigEmb(nn.Module):
    def __init__(self, dim, patch, grid):
        super().__init__()
        self.patch_embedding = nn.Conv2d(3, dim, patch, patch)
        self.position_embedding = nn.Embedding(grid[0] * grid[1], dim)

class _SigHead(nn.Module):                      # multihead attention pooling (MAP)
    def __init__(self, dim, heads, hidden=None):
        super().__init__()
        self.probe = nn.Parameter(torch.zeros(1, 1, dim))
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.layernorm = nn.LayerNorm(dim, eps=1e-6); self.mlp = _SigMlp(dim, hidden or dim * 4)
    def forward(self, x):
        h = self.attention(self.probe.expand(x.shape[0], -1, -1).to(x.dtype), x, x, need_weights=False)[0]
        h = h + self.mlp(self.layernorm(h))
        return h[:, 0]

class _SigEncoder(nn.Module):
    def __init__(self, dim, heads, dps, hidden=None):
        super().__init__(); self.layers = nn.ModuleList([_SigLayer(dim, heads, d, hidden) for d in dps])

class SiglipViT(nn.Module):
    def __init__(self, dim, depth, heads, grid, patch, drop_path=0.0, parts=False, hidden=None):
        super().__init__()
        self.grid, self.dim, self.parts = grid, dim, parts
        self.embeddings = _SigEmb(dim, patch, grid)
        dps = [drop_path * i / max(1, depth - 1) for i in range(depth)]
        self.encoder = _SigEncoder(dim, heads, dps, hidden)
        self.post_layernorm = nn.LayerNorm(dim, eps=1e-6)
        self.head = _SigHead(dim, heads, hidden)
        self.out_dim = (2 + 3 * parts) * dim
    @property
    def blocks(self): return self.encoder.layers
    def embed(self, x):
        p = self.embeddings.patch_embedding(x).flatten(2).transpose(1, 2)
        x = p + self.embeddings.position_embedding.weight[None].to(p.dtype)
        for l in self.encoder.layers: x = l(x)
        return self.post_layernorm(x)
    def image_embeds(self, x):                      # giống SiglipModel.get_image_features
        return self.head(self.embed(x))
    def forward(self, x, return_tokens=False):
        x = self.embed(x)
        e = F.normalize(self.head(x).float(), dim=-1)
        f = torch.cat([e, x.mean(1).float()] + ([s.float() for s in _stripe_means(x, self.grid)] if self.parts else []), 1)
        return (f, x) if return_tokens else f

SIGLIP_CFG = {"siglip2_vitb16": ("google/siglip2-base-patch16-224", 768, 12, 12, 16),
              "siglip_vitb16":  ("google/siglip-base-patch16-224", 768, 12, 12, 16),
              # 11/10 G14: SigLIP2 So400m/16 (Apache-2.0; 27 khối, dim 1152, MLP 4304; pos_embed 16×16 nội suy về lưới ảnh). Thành viên thứ 2 khác họ DINO
              "siglip2_so400m16": ("google/siglip2-so400m-patch16-256", 1152, 27, 16, 16, 4304),
              "siglip2_dbg": (None, 64, 2, 2, 16, 96)}                    # chỉ để chạy thử code trên CPU

def siglip_state_from_hf(hf_sd, grid):
    sd = {k[len("vision_model."):]: v for k, v in hf_sd.items() if k.startswith("vision_model.")}
    sd.pop("embeddings.position_ids", None)
    pe = sd["embeddings.position_embedding.weight"]
    g, s = _resize_pos(pe, grid, antialias=False)        # giống interpolate_pos_encoding của HF
    sd["embeddings.position_embedding.weight"] = g.to(pe.dtype)
    return sd, s

def load_siglip_pretrained(vit, arch):
    from transformers import AutoModel
    hf = AutoModel.from_pretrained(SIGLIP_CFG[arch][0]).state_dict()
    sd, s = siglip_state_from_hf(hf, vit.grid)
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, (missing, unexpected)
    print(f"Loaded {SIGLIP_CFG[arch][0]} | pos_embed {s}x{s} -> {vit.grid[0]}x{vit.grid[1]}")

# ------------------------------ Decoder truy vấn thuộc tính (K11) ------------------------------
class _XAttn(nn.Module):
    # attention đa đầu viết bằng nn.Linear thường (quantize_dynamic xử lý được, không dùng nn.MultiheadAttention)
    def __init__(self, dim, heads):
        super().__init__(); self.h = heads
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim); self.v = nn.Linear(dim, dim); self.o = nn.Linear(dim, dim)
    def forward(self, q, kv):
        B, Nq, C = q.shape; Nk = kv.shape[1]; h = self.h
        q = self.q(q).view(B, Nq, h, C // h).transpose(1, 2)
        k = self.k(kv).view(B, Nk, h, C // h).transpose(1, 2)
        v = self.v(kv).view(B, Nk, h, C // h).transpose(1, 2)
        return self.o(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, Nq, C))

class QueryDecoder(nn.Module):
    # Mỗi thuộc tính 1 query: cross-attention lên patch token (tự tìm vùng ảnh liên quan), rồi self-attention giữa 40 query
    # (quan hệ giữa thuộc tính, như một đồ thị học được theo từng ảnh). Đầu ra cộng vào logit của head tuyến tính.
    # w = 0 lúc khởi tạo nên ban đầu model y hệt bản gốc.
    def __init__(self, dim, num_attr, heads):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, num_attr, dim) * 0.02)
        self.ln_q = nn.LayerNorm(dim); self.ln_kv = nn.LayerNorm(dim); self.cross = _XAttn(dim, heads)
        self.ln_s = nn.LayerNorm(dim); self.self_attn = _XAttn(dim, heads)
        self.ln_f = nn.LayerNorm(dim); self.mlp = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim))
        self.w = nn.Parameter(torch.zeros(num_attr, dim)); self.b = nn.Parameter(torch.zeros(num_attr))
    def forward(self, tok):
        kv = self.ln_kv(tok)
        h = self.query.expand(tok.shape[0], -1, -1)
        h = h + self.cross(self.ln_q(h), kv)
        s = self.ln_s(h); h = h + self.self_attn(s, s)
        h = h + self.mlp(self.ln_f(h))
        return (h.float() * self.w).sum(-1) + self.b

# ------------------------------ DINOv3 ViT (RoPE 2D, patch 16), khoá khớp transformers DINOv3ViTModel ------------------------------
DINO3_CFG = {"dinov3_vits16": (384, 12, 6, "facebook/dinov3-vits16-pretrain-lvd1689m"),
             "dinov3_vitb16": (768, 12, 12, "facebook/dinov3-vitb16-pretrain-lvd1689m"),
             "dinov3_vitl16": (1024, 24, 16, "facebook/dinov3-vitl16-pretrain-lvd1689m")}

def _rot_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def _v3_rope(gh, gw, head_dim, base, device):
    # cos / sin cho từng patch: toạ độ tâm patch đưa về [-1, 1] (như get_patches_center_coordinates, không augment khi train)
    ch = torch.arange(0.5, gh, dtype=torch.float32, device=device) / gh
    cw = torch.arange(0.5, gw, dtype=torch.float32, device=device) / gw
    coords = torch.stack(torch.meshgrid(ch, cw, indexing="ij"), -1).flatten(0, 1) * 2.0 - 1.0
    inv = 1.0 / base ** torch.arange(0, 1, 4 / head_dim, dtype=torch.float32, device=device)
    ang = (2 * math.pi * coords[:, :, None] * inv[None, None, :]).flatten(1, 2).tile(2)
    return ang.cos(), ang.sin()

class _V3Emb(nn.Module):
    def __init__(self, dim, n_reg, patch):
        super().__init__()
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))         # không dùng, chỉ để khớp checkpoint
        self.register_tokens = nn.Parameter(torch.zeros(1, n_reg, dim))
        self.patch_embeddings = nn.Conv2d(3, dim, patch, patch)

class _V3Attn(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); self.h = heads
        self.q_proj = nn.Linear(dim, dim); self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim); self.o_proj = nn.Linear(dim, dim)
    def forward(self, x, cos, sin, n_pre):
        B, N, C = x.shape; hd = C // self.h
        q, k, v = (f(x).view(B, N, self.h, hd).transpose(1, 2) for f in (self.q_proj, self.k_proj, self.v_proj))
        cos, sin = cos.to(q.dtype), sin.to(q.dtype)
        rope = lambda t: torch.cat([t[:, :, :n_pre], t[:, :, n_pre:] * cos + _rot_half(t[:, :, n_pre:]) * sin], 2)   # RoPE chỉ cho patch
        o = F.scaled_dot_product_attention(rope(q), rope(k), v)
        return self.o_proj(o.transpose(1, 2).reshape(B, N, C))

class _V3LS(nn.Module):
    def __init__(self, dim): super().__init__(); self.lambda1 = nn.Parameter(torch.ones(dim))
    def forward(self, x): return x * self.lambda1

class _V3Mlp(nn.Module):
    def __init__(self, dim, hidden): super().__init__(); self.up_proj = nn.Linear(dim, hidden); self.down_proj = nn.Linear(hidden, dim)
    def forward(self, x): return self.down_proj(F.gelu(self.up_proj(x)))

class _V3Layer(nn.Module):
    def __init__(self, dim, heads, dp, eps):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=eps); self.attention = _V3Attn(dim, heads); self.layer_scale1 = _V3LS(dim)
        self.norm2 = nn.LayerNorm(dim, eps=eps); self.mlp = _V3Mlp(dim, dim * 4); self.layer_scale2 = _V3LS(dim)
        self.dp = dp
    def forward(self, x, cos, sin, n_pre):
        x = x + _drop_path(self.layer_scale1(self.attention(self.norm1(x), cos, sin, n_pre)), self.dp, self.training)
        return x + _drop_path(self.layer_scale2(self.mlp(self.norm2(x))), self.dp, self.training)

class DinoV3ViT(nn.Module):
    # Cùng giao diện với DinoViT (features / forward / grid / out_dim / blocks); RoPE nên đổi kích thước ảnh không cần nội suy gì
    def __init__(self, dim, depth, heads, grid, n_reg=4, drop_path=0.0, parts=False, patch=16, eps=1e-5, theta=100.0):
        super().__init__()
        self.grid, self.n_reg, self.parts, self.patch, self.heads, self.theta = grid, n_reg, parts, patch, heads, theta
        self.embeddings = _V3Emb(dim, n_reg, patch)
        dps = [drop_path * i / max(1, depth - 1) for i in range(depth)]
        self.layer = nn.ModuleList([_V3Layer(dim, heads, d, eps) for d in dps])
        self.norm = nn.LayerNorm(dim, eps=eps)
        self.out_dim = (2 + 3 * parts) * dim
        self._rope = {}
    @property
    def blocks(self): return self.layer
    def features(self, x, taps=()):
        B, _, H, W = x.shape; self.grid = (H // self.patch, W // self.patch)
        key = (self.grid, str(x.device))
        if key not in self._rope: self._rope[key] = _v3_rope(*self.grid, self.norm.normalized_shape[0] // self.heads, self.theta, x.device)
        cos, sin = self._rope[key]; e = self.embeddings; n_pre = 1 + self.n_reg
        x = torch.cat([e.cls_token.expand(B, -1, -1), e.register_tokens.expand(B, -1, -1),
                       e.patch_embeddings(x).flatten(2).transpose(1, 2)], 1)
        mids = []
        for i, L_ in enumerate(self.layer):
            x = L_(x, cos, sin, n_pre)
            if i in taps: mids.append(x[:, n_pre:])
        x = self.norm(x); pt = x[:, n_pre:]
        f = torch.cat([x[:, 0], pt.mean(1)] + (_stripe_means(pt, self.grid) if self.parts else []), 1)
        return f, pt, mids
    def forward(self, x, return_tokens=False):
        f, pt, _ = self.features(x)
        return (f, pt) if return_tokens else f

def load_dinov3_pretrained(vit, repo):
    # Chỉ dùng lúc TRAIN (cần mạng, quyền tải repo gated, token trong biến môi trường HF_TOKEN). Server không gọi hàm này.
    import os
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    sd = load_file(hf_hub_download(repo, "model.safetensors", token=os.environ.get("HF_TOKEN") or None))
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, ("DINOv3 khoá không khớp", missing[:5], unexpected[:5])
    print(f"Loaded {repo} (DINOv3, {len(sd)} tensor)")

# ------------------------------ DINOv3 ViT-H+/16 (840M), khoá khớp timm vit_huge_plus_patch16_dinov3 (repo timm không gated) ------------------------------
# Như DinoV3ViT nhưng: qkv gộp không bias, layer scale gamma_1/2, MLP SwiGLU (fc1_g, fc1_x, fc2; ẩn 5120), tên khoá kiểu timm Eva.
# RoPE giống hệt DINOv3-L (theta 100, rotate-half, chỉ cho patch). Thứ tự token: cls, 4 register, patch.
DINO3_EVA_CFG = {"dinov3_vith16p": (1280, 32, 20, 5120, "timm/vit_huge_plus_patch16_dinov3.lvd1689m"),   # dim, depth, heads, ẩn MLP, repo
                 "dinov3_dbgeva": (64, 3, 2, 128, None)}                                                  # chỉ để chạy thử code trên CPU

class _EvaAttn(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); self.h = heads; self.qkv = nn.Linear(dim, dim * 3, bias=False); self.proj = nn.Linear(dim, dim)
    def forward(self, x, cos, sin, n_pre):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4).unbind(0)
        cos, sin = cos.to(q.dtype), sin.to(q.dtype)
        rope = lambda t: torch.cat([t[:, :, :n_pre], t[:, :, n_pre:] * cos + _rot_half(t[:, :, n_pre:]) * sin], 2)
        o = F.scaled_dot_product_attention(rope(q), rope(k), v)
        return self.proj(o.transpose(1, 2).reshape(B, N, C))

class _SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__(); self.fc1_g = nn.Linear(dim, hidden); self.fc1_x = nn.Linear(dim, hidden); self.fc2 = nn.Linear(hidden, dim)
    def forward(self, x): return self.fc2(F.silu(self.fc1_g(x)) * self.fc1_x(x))

class _EvaBlock(nn.Module):
    def __init__(self, dim, heads, hidden, dp, eps):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=eps); self.attn = _EvaAttn(dim, heads); self.gamma_1 = nn.Parameter(torch.ones(dim))
        self.norm2 = nn.LayerNorm(dim, eps=eps); self.mlp = _SwiGLU(dim, hidden); self.gamma_2 = nn.Parameter(torch.ones(dim))
        self.dp = dp
    def forward(self, x, cos, sin, n_pre):
        x = x + _drop_path(self.gamma_1 * self.attn(self.norm1(x), cos, sin, n_pre), self.dp, self.training)
        return x + _drop_path(self.gamma_2 * self.mlp(self.norm2(x)), self.dp, self.training)

class DinoV3Eva(nn.Module):
    # Cùng giao diện với DinoV3ViT (features / forward / grid / out_dim / blocks / norm). ckpt=True: gradient checkpointing cho khối còn train
    def __init__(self, dim, depth, heads, hidden, grid, n_reg=4, drop_path=0.0, parts=False, patch=16, eps=1e-5, theta=100.0):
        super().__init__()
        self.grid, self.n_reg, self.parts, self.patch, self.heads, self.theta, self.ckpt = grid, n_reg, parts, patch, heads, theta, False
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim)); self.reg_token = nn.Parameter(torch.zeros(1, n_reg, dim))
        self.patch_embed = nn.Module(); self.patch_embed.proj = nn.Conv2d(3, dim, patch, patch)
        dps = [drop_path * i / max(1, depth - 1) for i in range(depth)]
        self.blocks = nn.ModuleList([_EvaBlock(dim, heads, hidden, d, eps) for d in dps])
        self.norm = nn.LayerNorm(dim, eps=eps)
        self.out_dim = (2 + 3 * parts) * dim
        self._rope = {}
    def tokens(self, x, taps=()):
        B, _, H, W = x.shape; self.grid = (H // self.patch, W // self.patch)
        key = (self.grid, str(x.device))
        if key not in self._rope: self._rope[key] = _v3_rope(*self.grid, self.norm.normalized_shape[0] // self.heads, self.theta, x.device)
        cos, sin = self._rope[key]; n_pre = 1 + self.n_reg
        x = torch.cat([self.cls_token.expand(B, -1, -1), self.reg_token.expand(B, -1, -1),
                       self.patch_embed.proj(x).flatten(2).transpose(1, 2)], 1)
        mids = []
        for i, b in enumerate(self.blocks):
            if self.ckpt and self.training and torch.is_grad_enabled() and any(p_.requires_grad for p_ in b.parameters()):
                x = torch.utils.checkpoint.checkpoint(b, x, cos, sin, n_pre, use_reentrant=False)
            else: x = b(x, cos, sin, n_pre)
            if i in taps: mids.append(x[:, n_pre:])
        return self.norm(x), n_pre, mids
    def features(self, x, taps=()):
        x, n_pre, mids = self.tokens(x, taps); pt = x[:, n_pre:]
        f = torch.cat([x[:, 0], pt.mean(1)] + (_stripe_means(pt, self.grid) if self.parts else []), 1)
        return f, pt, mids
    def forward(self, x, return_tokens=False):
        f, pt, _ = self.features(x)
        return (f, pt) if return_tokens else f

_EVA_CHECKED = set()
def load_dinov3_eva_pretrained(vit, repo, timm_name=None):
    # Chỉ dùng lúc TRAIN (cần mạng; repo timm không gated). Lần nạp đầu: so đầu ra với timm (nếu có) trên ảnh ngẫu nhiên, fp32 CPU.
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    sd = load_file(hf_hub_download(repo, "model.safetensors"))
    sd = {k: v for k, v in sd.items() if not k.startswith(("head.", "fc_norm.", "rope."))}
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, ("DINOv3 Eva khoá không khớp", missing[:5], unexpected[:5])
    print(f"Loaded {repo} (DINOv3 Eva, {len(sd)} tensor, {sum(v.numel() for v in sd.values()) / 1e6:.1f}M)")
    if repo in _EVA_CHECKED: return
    _EVA_CHECKED.add(repo)
    try:
        import timm
        tm = timm.create_model(timm_name or repo.split("/")[-1].split(".")[0], pretrained=False, num_classes=0, dynamic_img_size=True)
        tm.load_state_dict(sd, strict=False); tm.eval()
        g0, was = vit.grid, vit.training; vit.eval()
        with torch.no_grad():
            x = torch.randn(1, 3, 256, 128)
            ref = tm.forward_features(x); ours, n_pre, _ = vit.tokens(x)
        vit.grid = g0; vit.train(was)
        err = float((ref - ours).abs().max()); rel = err / float(ref.abs().max())
        print(f"[DINOv3 Eva] so với timm {timm.__version__}: lệch tuyệt đối max {err:.2e}, tương đối {rel:.2e}")
        assert rel < 1e-3, "DINOv3 Eva KHÔNG khớp timm"
        del tm
    except AssertionError: raise
    except Exception as e:
        print(f"[DINOv3 Eva] bỏ qua so sánh timm ({type(e).__name__}: {e})")

# ------------------------------ C-RADIOv4 (NVIDIA, 01/2026; NVIDIA Open Model License, weights HF không gated) ------------------------------
# 11/10: backbone chưng cất từ SigLIP2-g + DINOv3-7B + SAM3 (IN kNN 86.6 > DINOv3-H+ 85.8). Kiến trúc = timm vit_huge_patch16 (qkv có bias, MLP GELU,
# LN eps 1e-6, không layer scale, KHÔNG có norm cuối) + CPE: pos_embed học được trên lưới 128×128; khi suy luận nội suy song tuyến về
# (max(h,w))² rồi cắt góc trên-trái h×w (đúng ViTPatchGenerator._get_pos_embeddings, nhánh eval). Patch embed = Linear (c, yy, xx) không bias.
# 10 token đầu = 3 CLS theo teacher + 7 register; đặc trưng tóm tắt = CLS ở summary_idxs (2 teacher có summary). Đầu vào chuẩn hoá CLIP OpenAI.
RADIO_CFG = {"radio_v4h": (1280, 32, 16, 5120, 10, 128, "nvidia/C-RADIOv4-H"),      # dim, depth, heads, ẩn MLP, số token đầu, lưới pos, repo
             "radio_dbg": (64, 3, 2, 128, 10, 16, None)}                              # chỉ để chạy thử code trên CPU

class _VitAttn(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); self.h = heads; self.qkv = nn.Linear(dim, dim * 3); self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4).unbind(0)
        return self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, C))

class _VitMlp(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__(); self.fc1 = nn.Linear(dim, hidden); self.fc2 = nn.Linear(hidden, dim)
    def forward(self, x): return self.fc2(F.gelu(self.fc1(x)))

class _VitBlock(nn.Module):
    def __init__(self, dim, heads, hidden, dp, eps):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=eps); self.attn = _VitAttn(dim, heads)
        self.norm2 = nn.LayerNorm(dim, eps=eps); self.mlp = _VitMlp(dim, hidden); self.dp = dp
    def forward(self, x):
        x = x + _drop_path(self.attn(self.norm1(x)), self.dp, self.training)
        return x + _drop_path(self.mlp(self.norm2(x)), self.dp, self.training)

class RadioViT(nn.Module):
    # Cùng giao diện với DinoV3Eva (tokens / features / forward / grid / out_dim / blocks / norm / ckpt).
    def __init__(self, dim, depth, heads, hidden, n_pre, pos_grid, grid, drop_path=0.0, parts=False, patch=16, eps=1e-6, n_sum=2):
        super().__init__()
        self.grid, self.n_pre, self.parts, self.patch, self.pos_grid, self.ckpt = grid, n_pre, parts, patch, pos_grid, False
        self.cls_token = nn.Parameter(torch.zeros(n_pre, dim))                      # 3 CLS theo teacher + 7 register
        self.patch_embed = nn.Linear(3 * patch * patch, dim, bias=False)
        self.pos_embed = nn.Parameter(torch.zeros(1, pos_grid * pos_grid, dim))
        self.register_buffer("summary_idxs", torch.arange(n_sum, dtype=torch.long))
        dps = [drop_path * i / max(1, depth - 1) for i in range(depth)]
        self.blocks = nn.ModuleList([_VitBlock(dim, heads, hidden, d, eps) for d in dps])
        self.norm = nn.Identity()
        self.out_dim = (n_sum + 1 + 3 * parts) * dim
        self._pos = {}
    def pos(self, gh, gw, dtype):
        # nhánh eval của CPE; khi train tính lại mỗi lần (pos_embed còn học), khi eval lưu đệm theo lưới
        key = (gh, gw, str(self.pos_embed.device), dtype)
        if not self.training and key in self._pos: return self._pos[key]
        P, C = self.pos_grid, self.pos_embed.shape[-1]; m = max(gh, gw)
        pe = self.pos_embed.reshape(1, P, P, C).permute(0, 3, 1, 2)
        if (gh, gw) != (P, P):
            pe = F.interpolate(pe.float(), size=(m, m), mode="bilinear", align_corners=False)[..., :gh, :gw]
        pe = pe.flatten(2).transpose(1, 2).to(dtype)
        if not self.training: self._pos[key] = pe
        else: self._pos.clear()
        return pe
    def tokens(self, x, taps=()):
        B, _, H, W = x.shape; p = self.patch; self.grid = (H // p, W // p)
        t = F.conv2d(x, self.patch_embed.weight.reshape(-1, 3, p, p).to(x.dtype), stride=p).flatten(2).transpose(1, 2)
        t = t + self.pos(*self.grid, t.dtype)
        x = torch.cat([self.cls_token.unsqueeze(0).expand(B, -1, -1).to(t.dtype), t], 1)
        mids = []
        for i, b in enumerate(self.blocks):
            if self.ckpt and self.training and torch.is_grad_enabled() and any(p_.requires_grad for p_ in b.parameters()):
                x = torch.utils.checkpoint.checkpoint(b, x, use_reentrant=False)
            else: x = b(x)
            if i in taps: mids.append(x[:, self.n_pre:])
        return self.norm(x), self.n_pre, mids
    def features(self, x, taps=()):
        x, n_pre, mids = self.tokens(x, taps); pt = x[:, n_pre:]
        f = torch.cat([x[:, self.summary_idxs].flatten(1), pt.mean(1)] + (_stripe_means(pt, self.grid) if self.parts else []), 1)
        return f, pt, mids
    def forward(self, x, return_tokens=False):
        f, pt, _ = self.features(x)
        return (f, pt) if return_tokens else f

_RADIO_CHECKED = set()
def load_radio_pretrained(vit, repo):
    # Chỉ dùng lúc TRAIN (cần mạng; repo không gated). Khoá HF: radio_model.model.{blocks.*, patch_generator.{cls_token.token, embedder.weight, pos_embed}},
    # radio_model.summary_idxs, radio_model.input_conditioner.{norm_mean, norm_std}. Lần nạp đầu: so với code gốc NVIDIA (torch.hub) nếu tải được.
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    raw = load_file(hf_hub_download(repo, "model.safetensors")); sd = {}
    for k, v in raw.items():
        if k.startswith("radio_model.model.blocks."): sd[k[len("radio_model.model."):]] = v
        elif k == "radio_model.model.patch_generator.cls_token.token": sd["cls_token"] = v
        elif k == "radio_model.model.patch_generator.embedder.weight": sd["patch_embed.weight"] = v
        elif k == "radio_model.model.patch_generator.pos_embed": sd["pos_embed"] = v
        elif k == "radio_model.summary_idxs": sd["summary_idxs"] = v.long()
    mean, std = raw["radio_model.input_conditioner.norm_mean"].flatten(), raw["radio_model.input_conditioner.norm_std"].flatten()
    assert torch.allclose(mean, torch.tensor(CLIP_NORM[0]), atol=1e-4) and torch.allclose(std, torch.tensor(CLIP_NORM[1]), atol=1e-4), ("RADIO chuẩn hoá khác CLIP", mean, std)
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, ("RADIO khoá không khớp", missing[:5], unexpected[:5])
    print(f"Loaded {repo} (C-RADIO, {len(sd)} tensor, {sum(v.numel() for v in sd.values()) / 1e6:.1f}M, summary_idxs {sd['summary_idxs'].tolist()})")
    if repo in _RADIO_CHECKED: return
    _RADIO_CHECKED.add(repo)
    try:   # code gốc qua torch.hub (cần timm + einops; bản transformers đòi thêm open_clip). Đã kiểm local 11/10: lệch ≤ 1.1e-5 ở 256/224/192
        ref = torch.hub.load("NVlabs/RADIO", "radio_model", version="c-radio_v4-h", progress=False, skip_validation=True, trust_repo=True)
        ref.load_state_dict({k[len("radio_model."):]: v for k, v in raw.items() if k.startswith("radio_model.")}, strict=False); ref = ref.float().eval()
        g0, was = vit.grid, vit.training; vit.eval()
        with torch.no_grad():
            x = torch.rand(1, 3, 256, 128)
            out = ref(x); summ, feat = (out.summary, out.features) if hasattr(out, "summary") else (out[0], out[1])
            mn, sd_ = torch.tensor(CLIP_NORM[0]).view(1, 3, 1, 1), torch.tensor(CLIP_NORM[1]).view(1, 3, 1, 1)
            tok, n_pre, _ = vit.tokens((x - mn) / sd_)
            ours_s, ours_f = tok[:, vit.summary_idxs].flatten(1), tok[:, n_pre:]
        vit.grid = g0; vit.train(was)
        rel = max(float((summ - ours_s).abs().max() / summ.abs().max()), float((feat - ours_f).abs().max() / feat.abs().max()))
        print(f"[C-RADIO] so với code NVIDIA: lệch tương đối max {rel:.2e}")
        assert rel < 1e-3, "C-RADIO KHÔNG khớp code gốc"
        del ref
    except AssertionError: raise
    except Exception as e:
        print(f"[C-RADIO] bỏ qua so sánh code gốc ({type(e).__name__}: {e})")

# ------------------------------ PE-Core (Meta Perception Encoder, Apache-2.0), khoá khớp timm vit_pe_core_* ------------------------------
# Như timm Eva cấu hình PE: cls + pos_embed tuyệt đối + norm_pre, khối qkv gộp với RoPE 2D kiểu "cat" (xen kẽ, chỉ cho patch, lưới lệch 1,
# quy về lưới pretrain 14×14), MLP GELU, không layer scale; norm cuối; pooling bằng attention (1 latent, 8 head). Patch embed không bias.
PE_CFG = {"pe_core_b16": ("timm/vit_pe_core_base_patch16_224.fb", 768, 12, 12, 16, 14),    # repo, dim, depth, heads, patch, lưới pretrain
          "pe_core_l14": ("timm/vit_pe_core_large_patch14_336.fb", 1024, 24, 16, 14, 24)}
PE_NORM = ((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

def _pe_rope(gh, gw, head_dim, ref, device, temp=10000.0, offset=1.0):
    # = timm build_rotary_pos_embed(in_pixels=False, grid_indexing="xy", grid_offset=1, ref_feat_shape=(ref, ref)) -> sin, cos (N, head_dim)
    nb = head_dim // 4
    bands = 1.0 / temp ** (torch.arange(nb, dtype=torch.float32, device=device) / nb)
    tx = (torch.arange(gw, dtype=torch.float32, device=device) + offset) / gw * ref
    ty = (torch.arange(gh, dtype=torch.float32, device=device) + offset) / gh * ref
    gx, gy = torch.meshgrid(tx, ty, indexing="xy")
    pos = torch.stack([gx, gy], -1).unsqueeze(-1) * bands
    return pos.sin().reshape(gh * gw, -1).repeat_interleave(2, -1), pos.cos().reshape(gh * gw, -1).repeat_interleave(2, -1)

def _pe_rot(x):
    return torch.stack([-x[..., 1::2], x[..., ::2]], -1).reshape(x.shape)

class _PEAttn(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); self.h = heads; self.qkv = nn.Linear(dim, dim * 3); self.proj = nn.Linear(dim, dim)
    def forward(self, x, sin, cos):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4).unbind(0)
        sin, cos = sin.to(q.dtype), cos.to(q.dtype)
        rope = lambda t: torch.cat([t[:, :, :1], t[:, :, 1:] * cos + _pe_rot(t[:, :, 1:]) * sin], 2)   # RoPE chỉ cho patch (bỏ cls)
        o = F.scaled_dot_product_attention(rope(q), rope(k), v)
        return self.proj(o.transpose(1, 2).reshape(B, N, C))

class _PEMlp(nn.Module):
    def __init__(self, dim, hidden): super().__init__(); self.fc1 = nn.Linear(dim, hidden); self.fc2 = nn.Linear(hidden, dim)
    def forward(self, x): return self.fc2(F.gelu(self.fc1(x)))

class _PEBlock(nn.Module):
    def __init__(self, dim, heads, dp):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-5); self.attn = _PEAttn(dim, heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-5); self.mlp = _PEMlp(dim, dim * 4); self.dp = dp
    def forward(self, x, sin, cos):
        x = x + _drop_path(self.attn(self.norm1(x), sin, cos), self.dp, self.training)
        return x + _drop_path(self.mlp(self.norm2(x)), self.dp, self.training)

class _PEPool(nn.Module):
    # = timm AttentionPoolLatent (1 latent, pool "token")
    def __init__(self, dim, heads=8):
        super().__init__(); self.h = heads
        self.latent = nn.Parameter(torch.zeros(1, 1, dim)); self.q = nn.Linear(dim, dim); self.kv = nn.Linear(dim, dim * 2)
        self.proj = nn.Linear(dim, dim); self.norm = nn.LayerNorm(dim, eps=1e-5); self.mlp = _PEMlp(dim, dim * 4)
    def forward(self, x):
        B, N, C = x.shape; hd = C // self.h
        q = self.q(self.latent.expand(B, -1, -1)).reshape(B, 1, self.h, hd).transpose(1, 2)
        k, v = self.kv(x).reshape(B, N, 2, self.h, hd).permute(2, 0, 3, 1, 4).unbind(0)
        o = self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, 1, C))
        o = o + self.mlp(self.norm(o))
        return o[:, 0]

class _PEPatch(nn.Module):
    def __init__(self, dim, patch): super().__init__(); self.proj = nn.Conv2d(3, dim, patch, patch, bias=False)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class PECoreViT(nn.Module):
    # Cùng giao diện với DinoV3ViT (features / forward / grid / out_dim / blocks / norm). Đặc trưng = [pooling attention ; TB patch].
    # pos_embed tuyệt đối theo lưới lúc tạo model -> mọi ảnh phải đúng kích thước đó (pipeline luôn resize về IMG).
    def __init__(self, dim, depth, heads, grid, patch=16, ref=14, drop_path=0.0, parts=False):
        super().__init__()
        self.grid, self.patch, self.heads, self.ref, self.parts = tuple(grid), patch, heads, ref, parts
        self.patch_embed = _PEPatch(dim, patch)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + grid[0] * grid[1], dim))
        self.norm_pre = nn.LayerNorm(dim, eps=1e-5)
        dps = [drop_path * i / max(1, depth - 1) for i in range(depth)]
        self.blocks = nn.ModuleList([_PEBlock(dim, heads, dps[i]) for i in range(depth)])
        self.norm = nn.LayerNorm(dim, eps=1e-5); self.attn_pool = _PEPool(dim)
        self.out_dim = dim * (2 + (3 if parts else 0)); self._rope = {}
    def features(self, x, taps=()):
        B, _, H, W = x.shape; g = (H // self.patch, W // self.patch)
        assert g == self.grid, f"PE-Core: lưới ảnh {g} khác lưới lúc tạo model {self.grid}"
        key = str(x.device)
        if key not in self._rope: self._rope[key] = _pe_rope(*g, self.pos_embed.shape[-1] // self.heads, self.ref, x.device)
        sin, cos = self._rope[key]
        x = torch.cat([self.cls_token.expand(B, -1, -1), self.patch_embed(x)], 1) + self.pos_embed
        x = self.norm_pre(x); mids = []
        for i, blk in enumerate(self.blocks):
            x = blk(x, sin, cos)
            if i in taps: mids.append(x[:, 1:])
        x = self.norm(x); pt = x[:, 1:]
        f = torch.cat([self.attn_pool(x), pt.mean(1)] + (_stripe_means(pt, self.grid) if self.parts else []), 1)
        return f, pt, mids
    def forward(self, x, return_tokens=False):
        f, pt, _ = self.features(x)
        return (f, pt) if return_tokens else f

def load_pe_pretrained(vit, repo):
    # Chỉ dùng lúc TRAIN (cần mạng). pos_embed pretrain (lưới 14×14) nội suy bicubic + antialias sang lưới hiện tại như timm resample_abs_pos_embed.
    import os
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    sd = load_file(hf_hub_download(repo, "model.safetensors", token=os.environ.get("HF_TOKEN") or None))
    sd = {k: v for k, v in sd.items() if not k.startswith(("head.", "fc_norm."))}
    pe = sd["pos_embed"]; n0 = int(round((pe.shape[1] - 1) ** 0.5)); gh, gw = vit.grid
    if (n0, n0) != (gh, gw):
        g = pe[:, 1:].reshape(1, n0, n0, -1).permute(0, 3, 1, 2).float()
        g = F.interpolate(g, size=(gh, gw), mode="bicubic", antialias=True, align_corners=False)
        sd["pos_embed"] = torch.cat([pe[:, :1].float(), g.permute(0, 2, 3, 1).reshape(1, gh * gw, -1)], 1).to(pe.dtype)
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, ("PE-Core khoá không khớp", missing[:5], unexpected[:5])
    print(f"Loaded {repo} (PE-Core, {len(sd)} tensor, pos_embed {n0}x{n0} -> {gh}x{gw})")

# ------------------------------ E12: nhánh màu ------------------------------
# Lý do: DINOv2 được pretrain với ColorJitter(hue 0.1, saturation 0.2, p 0.8) + 20 % ảnh xám, nên đặc trưng cuối được dạy để bất biến
# với màu (L tăng mọi nhóm trừ màu). Hai nhánh cộng thêm logit cho 24 thuộc tính màu, khởi tạo bằng 0 (lúc đầu = model gốc).
UB_COLOR = list(range(8, 20))       # UpperBody-Color-* theo thứ tự 40 thuộc tính của BTC
LB_COLOR = list(range(21, 33))      # LowerBody-Color-*

class MidColorHead(nn.Module):
    # Head màu từ patch token của các khối giữa (còn giữ thông tin màu), gộp theo dải thân trên / thân dưới
    def __init__(self, dim, n_taps, n_attr=40):
        super().__init__()
        self.n_attr = n_attr
        self.norm_u, self.norm_l = nn.LayerNorm(dim * n_taps), nn.LayerNorm(dim * n_taps)
        self.ub, self.lb = nn.Linear(dim * n_taps, len(UB_COLOR)), nn.Linear(dim * n_taps, len(LB_COLOR))
        for l_ in (self.ub, self.lb): nn.init.zeros_(l_.weight); nn.init.zeros_(l_.bias)
        self.register_buffer("iu", torch.tensor(UB_COLOR), persistent=False)
        self.register_buffer("il", torch.tensor(LB_COLOR), persistent=False)
    def forward(self, mids, grid):
        st = [_stripe_means(t, grid) for t in mids]
        u = self.ub(self.norm_u(torch.cat([a[1] for a in st], 1))); l_ = self.lb(self.norm_l(torch.cat([a[2] for a in st], 1)))
        return u.new_zeros(u.shape[0], self.n_attr).index_copy(1, self.iu, u).index_copy(1, self.il, l_)

class ColorStats(nn.Module):
    # Thống kê màu tường minh: histogram mềm trên lưới k×k×k màu RGB của pixel thật (đã bỏ chuẩn hoá), vùng áo và vùng quần
    # (bỏ 20 % mỗi bên để bớt nền), qua MLP nhỏ. Rẻ: ảnh được avg-pool 4×4 trước khi tính.
    def __init__(self, mean, std, k=4, hidden=128, n_attr=40):
        super().__init__()
        g = torch.linspace(0, 1, k)
        self.register_buffer("c", torch.stack(torch.meshgrid(g, g, g, indexing="ij"), -1).reshape(-1, 3), persistent=False)
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1), persistent=False)
        self.inv_tau, self.n_attr = 2.0 * (k - 1) ** 2, n_attr            # 1 / (2σ²), σ = nửa khoảng cách lưới
        self.mu = nn.Sequential(nn.Linear(k ** 3, hidden), nn.GELU(), nn.Linear(hidden, len(UB_COLOR)))
        self.ml = nn.Sequential(nn.Linear(k ** 3, hidden), nn.GELU(), nn.Linear(hidden, len(LB_COLOR)))
        for l_ in (self.mu[2], self.ml[2]): nn.init.zeros_(l_.weight); nn.init.zeros_(l_.bias)
        self.register_buffer("iu", torch.tensor(UB_COLOR), persistent=False)
        self.register_buffer("il", torch.tensor(LB_COLOR), persistent=False)
    def hist(self, img):
        px = img.flatten(2).transpose(1, 2)                                 # (B, P, 3)
        d2 = ((px.unsqueeze(2) - self.c) ** 2).sum(-1)                      # (B, P, k³)
        return torch.softmax(-d2 * self.inv_tau, -1).mean(1).sqrt()
    def forward(self, x):
        img = F.avg_pool2d((x.float() * self.std + self.mean).clamp(0, 1), 4)
        H, W = img.shape[-2:]; c0 = int(W * 0.2); c1 = max(int(W * 0.8), c0 + 1)
        u = self.mu(self.hist(img[:, :, int(H * 0.2):max(int(H * 0.6), int(H * 0.2) + 1), c0:c1]))
        l_ = self.ml(self.hist(img[:, :, int(H * 0.5):, c0:c1]))
        return u.new_zeros(u.shape[0], self.n_attr).index_copy(1, self.iu, u).index_copy(1, self.il, l_.to(u.dtype))

# ------------------------------ Mô hình PAR ------------------------------
def patch_of(arch):
    return (CLIP_CFG[arch][4] if arch.startswith("clip") else SIGLIP_CFG[arch][4] if arch.startswith("siglip")
            else PE_CFG[arch][4] if arch.startswith("pe_core") else 16 if arch.startswith(("dinov3", "radio")) else 14)

class DinoPartMasker(nn.Module):
    # E27 (H8): DINO đã fine-tune, đóng băng, khoanh vùng áo / quần cho model khác. Logit màu theo từng patch fc([CLS ; token_i]);
    # điểm vùng = max logit 12 màu áo / quần; mask = softmax(điểm / τ) trong khung dọc (áo 10–70 %, quần 40–100 % chiều cao).
    # fixed=True: mask đều trong cùng khung (đối chứng, không cần DINO). Ảnh vào đã chuẩn hoá theo model chính, đổi sang đầu vào của DINO.
    def __init__(self, pm, in_norm):
        super().__init__()
        self.tau, self.fixed = float(pm.get("TAU", 1.5)), bool(pm.get("FIXED", False))
        self.size = (int(pm.get("src_h", 256)), int(pm.get("src_w", 128)))
        self.net = None if self.fixed else PARNet(pm["src_arch"], pretrained=False, img_h=self.size[0], img_w=self.size[1])
        sm = pm.get("src_mean", IMAGENET_NORM[0]); ss = pm.get("src_std", IMAGENET_NORM[1])
        for n_, v in (("mi", in_norm[0]), ("si", in_norm[1]), ("md", sm), ("sd", ss)):
            self.register_buffer(n_, torch.tensor(v, dtype=torch.float32).view(1, 3, 1, 1), persistent=False)
    def train(self, mode=True):
        return super().train(False)                      # luôn ở chế độ eval (không dropout / drop path)
    @torch.no_grad()
    def forward(self, x, grid):
        B = x.shape[0]
        if self.fixed:
            GH, GW = 16, 8; su = sl = x.new_zeros(B, GH * GW, dtype=torch.float32)
        else:
            xd = F.interpolate(x.float() * self.si + self.mi, size=self.size, mode="bilinear", align_corners=False)
            fe, pt, _ = self.net.backbone.features((xd - self.md) / self.sd)
            GH, GW = self.net.backbone.grid; N, C = pt.shape[1], pt.shape[2]
            zi = self.net.fc(torch.cat([fe[:, :C].unsqueeze(1).expand(B, N, C), pt], 2)).float()
            su, sl = zi[..., UB_COLOR].amax(-1), zi[..., LB_COLOR].amax(-1)
        rows = (torch.arange(GH, device=x.device, dtype=torch.float32) / GH).view(GH, 1).expand(GH, GW).reshape(-1)
        pu = torch.where((rows >= 0.10) & (rows < 0.70), 0.0, -1e4); pl = torch.where(rows >= 0.40, 0.0, -1e4)
        def shrink(m_):
            m_ = F.adaptive_avg_pool2d(m_.view(B, 1, GH, GW), tuple(grid)).flatten(1)
            return m_ / m_.sum(1, keepdim=True).clamp(min=1e-12)
        return shrink(torch.softmax(su / self.tau + pu, 1)), shrink(torch.softmax(sl / self.tau + pl, 1))

class PartMaskColorHead(nn.Module):
    # E27: token của model chính gộp theo mask áo / quần -> logit 12 màu áo / 12 màu quần (lớp cuối khởi tạo 0 = lúc đầu như model gốc)
    def __init__(self, dim, n_src, n_attr=40):
        super().__init__()
        self.n_attr = n_attr
        self.nu, self.nl = nn.LayerNorm(dim * n_src), nn.LayerNorm(dim * n_src)
        self.ub, self.lb = nn.Linear(dim * n_src, len(UB_COLOR)), nn.Linear(dim * n_src, len(LB_COLOR))
        for l_ in (self.ub, self.lb): nn.init.zeros_(l_.weight); nn.init.zeros_(l_.bias)
        self.register_buffer("iu", torch.tensor(UB_COLOR), persistent=False)
        self.register_buffer("il", torch.tensor(LB_COLOR), persistent=False)
    def forward(self, toks, mu, ml):
        u = torch.cat([torch.einsum("bn,bnc->bc", mu.to(t.dtype), t) for t in toks], 1).float()
        l_ = torch.cat([torch.einsum("bn,bnc->bc", ml.to(t.dtype), t) for t in toks], 1).float()
        u, l_ = self.ub(self.nu(u)), self.lb(self.nl(l_))
        return u.new_zeros(u.shape[0], self.n_attr).index_copy(1, self.iu, u).index_copy(1, self.il, l_)

class PARNet(nn.Module):
    def __init__(self, arch="dinov2_vitb14_reg", num_attr=40, pretrained=False, drop=0.2,
                 img_h=252, img_w=126, drop_path=0.0, parts=False, qdec=False, cmid=(), chist=False, pmask=None):
        super().__init__()
        self.arch, self.parts = arch, parts
        self.qdec = None                                   # K11: QueryDecoder nếu qdec=True
        self.cmid, self.chead, self.cstats = tuple(cmid), None, None      # E12: nhánh màu (chỉ DINOv2)
        self.pmasker, self.phead, self.ptaps = None, None, tuple((pmask or {}).get("TAPS", ()))   # E27: màu theo vùng do DINO khoanh
        if arch.startswith(("dinov2", "dinov3", "pe_core", "radio")):
            if arch.startswith("pe_core"):                 # PE-Core: patch 16, pos_embed tuyệt đối + RoPE 2D, pooling attention
                repo, dim, depth, heads, patch, ref = PE_CFG[arch]
                self.backbone = PECoreViT(dim, depth, heads, (img_h // patch, img_w // patch), patch, ref, drop_path, parts)
                if pretrained: load_pe_pretrained(self.backbone, repo)
            elif arch in RADIO_CFG:                        # C-RADIOv4 (11/10): ViT-H/16 + CPE, 10 token đầu, chuẩn hoá CLIP
                dim, depth, heads, hidden, n_pre, pgrid, repo = RADIO_CFG[arch]
                self.backbone = RadioViT(dim, depth, heads, hidden, n_pre, pgrid, (img_h // 16, img_w // 16), drop_path, parts)
                if pretrained and repo: load_radio_pretrained(self.backbone, repo)
            elif arch in DINO3_EVA_CFG:                    # DINOv3 ViT-H+ (khoá timm)
                dim, depth, heads, hidden, repo = DINO3_EVA_CFG[arch]
                self.backbone = DinoV3Eva(dim, depth, heads, hidden, (img_h // 16, img_w // 16), 4, drop_path, parts)
                if pretrained and repo: load_dinov3_eva_pretrained(self.backbone, repo)
            elif arch.startswith("dinov3"):                # DINOv3: patch 16, RoPE, 4 register
                dim, depth, heads, repo = DINO3_CFG[arch]
                self.backbone = DinoV3ViT(dim, depth, heads, (img_h // 16, img_w // 16), 4, drop_path, parts)
                if pretrained: load_dinov3_pretrained(self.backbone, repo)
            else:
                name = arch.replace("_reg", ""); n_reg = 4 if arch.endswith("_reg") else 0
                dim, depth, heads = DINO_CFG[name]
                self.backbone = DinoViT(dim, depth, heads, (img_h // 14, img_w // 14), n_reg, 14, drop_path, parts)
                if pretrained: load_dinov2_pretrained(self.backbone, name, n_reg)
            self.fc = nn.Sequential(nn.LayerNorm(self.backbone.out_dim), nn.Dropout(drop),
                                    nn.Linear(self.backbone.out_dim, num_attr))
            if qdec: self.qdec = QueryDecoder(dim, num_attr, dim // 64)
            if cmid: self.chead = MidColorHead(dim, len(cmid), num_attr)
            if chist: self.cstats = ColorStats(*norm_for(arch), n_attr=num_attr)
            if pmask:
                assert arch.startswith("pe_core") or not self.ptaps, "E27: token khối giữa chỉ có ở PE-Core / DINO"
                self.pmasker, self.phead = DinoPartMasker(pmask, norm_for(arch)), PartMaskColorHead(dim, 1 + len(self.ptaps), num_attr)
            return
        if arch.startswith("clip"):
            _, dim, depth, heads, patch, pdim = CLIP_CFG[arch]
            self.backbone = ClipViT(dim, depth, heads, (img_h // patch, img_w // patch), patch, pdim, drop_path, parts)
            if pretrained: load_clip_pretrained(self.backbone, arch)
            # Không LayerNorm ở head: phần đầu của feature là embedding CLIP đã chuẩn hoá, head khởi tạo bằng text embedding
            self.fc = nn.Sequential(nn.Identity(), nn.Dropout(drop), nn.Linear(self.backbone.out_dim, num_attr))
            if qdec: self.qdec = QueryDecoder(dim, num_attr, dim // 64)
            if pmask:
                assert not self.ptaps, "E27: CLIP chỉ dùng token khối cuối"
                self.pmasker, self.phead = DinoPartMasker(pmask, norm_for(arch)), PartMaskColorHead(dim, 1, num_attr)
            return
        if arch.startswith("siglip"):
            _, dim, depth, heads, patch = SIGLIP_CFG[arch][:5]; hid = SIGLIP_CFG[arch][5] if len(SIGLIP_CFG[arch]) > 5 else None
            self.backbone = SiglipViT(dim, depth, heads, (img_h // patch, img_w // patch), patch, drop_path, parts, hid)
            if pretrained and SIGLIP_CFG[arch][0]: load_siglip_pretrained(self.backbone, arch)
            self.fc = nn.Sequential(nn.Identity(), nn.Dropout(drop), nn.Linear(self.backbone.out_dim, num_attr))
            if qdec: self.qdec = QueryDecoder(dim, num_attr, dim // 64)
            return
        m = getattr(torchvision.models, arch)(weights="DEFAULT" if pretrained else None)
        if arch.startswith("convnext"):
            dim = m.classifier[2].in_features
            m.classifier[2] = nn.Sequential(nn.Dropout(drop), nn.Linear(dim, num_attr))
        elif arch.startswith("resnet"):
            dim = m.fc.in_features
            m.fc = nn.Sequential(nn.Dropout(drop), nn.Linear(dim, num_attr))
        else:
            raise ValueError(arch)
        self.net = m

    @property
    def head(self):
        if self.arch.startswith(("dinov2", "dinov3", "clip", "siglip", "pe_core", "radio")): return self.fc[2]
        return self.net.classifier[2][1] if self.arch.startswith("convnext") else self.net.fc[1]

    def forward(self, x, return_feat=False):
        # return_feat=True: trả thêm (đặc trưng gộp, patch token cuối) cho chưng cất đặc trưng (E11); mặc định chỉ trả logit như cũ
        if self.arch.startswith(("dinov2", "dinov3", "clip", "siglip", "pe_core", "radio")):
            mids = None
            if self.phead is not None:                     # E27: token gộp theo mask vùng của DINO
                if self.ptaps: f, tok, mids = self.backbone.features(x, self.ptaps)
                else: f, tok = self.backbone(x, return_tokens=True)
                z = self.fc(f) + self.phead([tok] + list(mids or []), *self.pmasker(x, self.backbone.grid))
                return (z, f, tok) if return_feat else z
            if self.chead is not None: f, tok, mids = self.backbone.features(x, self.cmid)
            elif self.qdec is not None or return_feat: f, tok = self.backbone(x, return_tokens=True)
            else: f, tok = self.backbone(x), None
            z = self.fc(f)
            if self.qdec is not None: z = z + self.qdec(tok)
            if self.chead is not None: z = z + self.chead(mids, self.backbone.grid)
            if self.cstats is not None: z = z + self.cstats(x)
            return (z, f, tok) if return_feat else z
        return self.net(x)

In [ ]:
print("[cell] torch", flush=True)
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
upar_model = sys.modules["model"]                 # module do cell %%module model tạo (trong bộ nhớ)
PARNet, preprocess, norm_for = upar_model.PARNet, upar_model.preprocess, upar_model.norm_for

# DINOv3-L (model OOF): repo facebook/* là gated. Có HF_TOKEN thì tải bản gốc, không thì lấy bản timm công khai rồi đổi tên khoá
# (cùng weights). Hàm này thay model.load_dinov3_pretrained khi TRAIN; server chấm không gọi hàm nạp pretrained nào.
DINO3_TIMM = {"facebook/dinov3-vits16-pretrain-lvd1689m": "timm/vit_small_patch16_dinov3.lvd1689m",
              "facebook/dinov3-vitb16-pretrain-lvd1689m": "timm/vit_base_patch16_dinov3.lvd1689m",
              "facebook/dinov3-vitl16-pretrain-lvd1689m": "timm/vit_large_patch16_dinov3.lvd1689m"}
def dinov3_sd_from_timm(t):
    sd = {"embeddings.cls_token": t["cls_token"], "embeddings.register_tokens": t["reg_token"],
          "embeddings.patch_embeddings.weight": t["patch_embed.proj.weight"], "embeddings.patch_embeddings.bias": t["patch_embed.proj.bias"],
          "embeddings.mask_token": torch.zeros_like(t["cls_token"]), "norm.weight": t["norm.weight"], "norm.bias": t["norm.bias"]}
    n = 1 + max(int(k.split(".")[1]) for k in t if k.startswith("blocks."))
    for i in range(n):
        b, o = f"blocks.{i}.", f"layer.{i}."
        q, k, v = t[b + "attn.qkv.weight"].chunk(3, 0)
        sd.update({o + "norm1.weight": t[b + "norm1.weight"], o + "norm1.bias": t[b + "norm1.bias"],
                   o + "attention.q_proj.weight": q, o + "attention.k_proj.weight": k, o + "attention.v_proj.weight": v,
                   o + "attention.q_proj.bias": torch.zeros(q.shape[0], dtype=q.dtype), o + "attention.v_proj.bias": torch.zeros(v.shape[0], dtype=v.dtype),
                   o + "attention.o_proj.weight": t[b + "attn.proj.weight"], o + "attention.o_proj.bias": t[b + "attn.proj.bias"],
                   o + "layer_scale1.lambda1": t[b + "gamma_1"], o + "layer_scale2.lambda1": t[b + "gamma_2"],
                   o + "norm2.weight": t[b + "norm2.weight"], o + "norm2.bias": t[b + "norm2.bias"],
                   o + "mlp.up_proj.weight": t[b + "mlp.fc1.weight"], o + "mlp.up_proj.bias": t[b + "mlp.fc1.bias"],
                   o + "mlp.down_proj.weight": t[b + "mlp.fc2.weight"], o + "mlp.down_proj.bias": t[b + "mlp.fc2.bias"]})
    return sd

def load_dinov3_pretrained_any(vit, repo):
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    tok = os.environ.get("HF_TOKEN") or None; sd = None
    if tok and not os.environ.get("UPAR_DINOV3_TIMM"):
        try: sd = load_file(hf_hub_download(repo, "model.safetensors", token=tok)); src = repo
        except Exception as e: print(f"[DINOv3] không tải được {repo} ({type(e).__name__}) -> dùng bản timm")
    if sd is None:
        src = DINO3_TIMM[repo]; sd = dinov3_sd_from_timm(load_file(hf_hub_download(src, "model.safetensors")))
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing and not unexpected, ("DINOv3 khoá không khớp", missing[:5], unexpected[:5])
    print(f"Loaded {src} (DINOv3, {len(sd)} tensor)")
upar_model.load_dinov3_pretrained = load_dinov3_pretrained_any

seed_all(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert dev.type == "cuda", "cần GPU (Kaggle: Settings -> Accelerator -> GPU T4 x2)"
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
sigm = lambda z: 1 / (1 + np.exp(-z))

def open_rgb(path, tries=7):
    # ổ Input gắn qua mạng đôi khi báo OSError tạm thời -> đọc lại có chờ
    for k in range(tries):
        try:
            with Image.open(path) as im: return im.convert("RGB")
        except OSError:
            if k == tries - 1: raise
            time.sleep(0.5 * 2 ** k)

print("Device:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "| torch", torch.__version__)

## 4. Tăng cường ảnh, dataset, loader

- Tăng cường ảnh như lúc train bản gốc: độ phân giải thấp (p 0.4), nén JPEG (p 0.3), lật ngang, pad 8 rồi crop, ColorJitter (p 0.8), GaussianBlur (p 0.15), RandomErasing (p 0.3).
- `CropJitter` / `MotionBlur` / `WBJitter` có p = 0 (không đổi ảnh) nhưng vẫn giữ trong chuỗi, như bản gốc.
- Lấy mẫu train có trọng số theo dataset: n_ds^-0.5 mỗi ảnh.

In [ ]:
print("[cell] augment + loader", flush=True)
class LowRes:
    """Thu nhỏ về chiều rộng ngẫu nhiên rồi phóng lại: mô phỏng crop giám sát độ phân giải thấp."""
    def __init__(self, p, wmin=20, wmax=64): self.p, self.wmin, self.wmax = p, wmin, wmax
    def __call__(self, img):
        if random.random() > self.p or img.width <= self.wmin: return img
        w = random.randint(self.wmin, min(self.wmax, img.width)); h = max(1, round(img.height * w / img.width))
        return img.resize((w, h), random.choice([0, 2, 3])).resize(img.size, random.choice([2, 3]))

class RandJPEG:
    def __init__(self, p, qmin=25, qmax=90): self.p, self.qmin, self.qmax = p, qmin, qmax
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=random.randint(self.qmin, self.qmax)); buf.seek(0)
        return Image.open(buf).convert("RGB")

class MotionBlur:
    """Mờ chuyển động 1 hướng ngẫu nhiên, kernel 3 hoặc 5."""
    def __init__(self, p): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        from PIL import ImageFilter
        k = random.choice([3, 5]); d = random.choice(["h", "v", "d1", "d2"]); w = [0.0] * (k * k)
        for i in range(k):
            r, c = {"h": (k // 2, i), "v": (i, k // 2), "d1": (i, i), "d2": (i, k - 1 - i)}[d]
            w[r * k + c] = 1.0
        return img.filter(ImageFilter.Kernel((k, k), w, scale=k))

class CropJitter:
    """Cắt một khung con ngẫu nhiên (cao/rộng độc lập trong [cmin, 1]) rồi mới resize."""
    def __init__(self, p, cmin=0.8): self.p, self.cmin = p, cmin
    def __call__(self, img):
        if random.random() > self.p: return img
        W0, H0 = img.size; sh_, sw_ = random.uniform(self.cmin, 1), random.uniform(self.cmin, 1)
        h, w = max(1, int(H0 * sh_)), max(1, int(W0 * sw_)); y, x = random.randint(0, H0 - h), random.randint(0, W0 - w)
        return img.crop((x, y, x + w, y + h))

class WBJitter:
    """Nhân mỗi kênh màu một hệ số ngẫu nhiên nhỏ (ám màu camera)."""
    def __init__(self, p, r=0.15): self.p, self.r = p, r
    def __call__(self, t):
        if random.random() > self.p: return t
        g = torch.empty(3).uniform_(1 - self.r, 1 + self.r); g = g / g.mean()
        return (t * g[:, None, None]).clamp(0, 1)

def make_train_tf(H, W, mean, std):
    return T.Compose([CropJitter(0.0, 0.8), LowRes(0.4, 20, 64), MotionBlur(0.0), RandJPEG(0.3, 25, 90),
                      T.Resize((H, W)), T.RandomHorizontalFlip(), T.Pad(8, padding_mode="edge"), T.RandomCrop((H, W)),
                      T.RandomApply([T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.1, hue=0.0)], p=0.8),
                      T.RandomApply([T.GaussianBlur(5, (0.1, 1.2))], p=0.15),
                      T.ToTensor(), WBJitter(0.0, 0.15), T.Normalize(mean, std),
                      T.RandomErasing(p=0.3, scale=(0.02, 0.12), value="random")])

class PARDataset(Dataset):
    def __init__(self, frame, H, W, mean, std, train, ym=None):
        self.paths = [str(DATA / p) for p in frame.image]
        d = frame.ds.map(DS_ID).values
        y, m = (frame[ATTR].values.astype(np.float32), MASK[d]) if ym is None else ym     # ym: (nhãn, mask) đã CMASK
        self.y, self.m, self.d = torch.as_tensor(y), torch.as_tensor(m), torch.as_tensor(d, dtype=torch.int64)
        self.train, self.H, self.W, self.mean, self.std = train, H, W, mean, std
        self.tf = make_train_tf(H, W, mean, std) if train else None
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = open_rgb(self.paths[i])
        x = self.tf(img) if self.train else preprocess(img, self.H, self.W, self.mean, self.std)
        return x, self.y[i], self.m[i], self.d[i]

def _worker_init(_): torch.set_num_threads(1)

def make_loader(frame, H, W, mean, std, train, batch, ym=None):
    ds = PARDataset(frame, H, W, mean, std, train, ym)
    kw = dict(num_workers=WORKERS, worker_init_fn=_worker_init, pin_memory=dev.type == "cuda")
    if WORKERS > 0: kw.update(prefetch_factor=2, persistent_workers=train)
    if not train: return DataLoader(ds, batch_size=batch, shuffle=False, **kw)
    n_ds = frame.ds.map(frame.ds.value_counts()).values.astype(np.float64); w = n_ds ** (-0.5)
    sampler = WeightedRandomSampler(torch.as_tensor(w / w.sum()), num_samples=len(frame), replacement=True)
    return DataLoader(ds, batch_size=batch, sampler=sampler, drop_last=True, **kw)

## 5. Loss, metric, hiệu chỉnh ngưỡng

- **Loss**: BCE có trọng số theo tỷ lệ dương (e^(1−p) cho nhãn dương, e^p cho nhãn âm), làm mượt nhãn 0.05. Chỉ tính trên ô có mask = 1.
- **Metric**: như scorer BTC, Challenge_Avg = trung bình điều hoà của mA và instance-F1, lấy trung bình theo dataset.
- **Hiệu chỉnh ngưỡng**: coordinate descent từng thuộc tính, lưới 0.05–0.95 bước 0.025, 2 lượt.

In [ ]:
print("[cell] loss + metrics", flush=True)
class MaskedLoss(nn.Module):
    def __init__(self, pos_rate, smooth=0.05):
        super().__init__()
        pr = torch.as_tensor(pos_rate)
        self.register_buffer("wp", torch.exp(1 - pr)); self.register_buffer("wn", torch.exp(pr))
        self.smooth = smooth
    def forward(self, logits, y, m):
        t = y * (1 - self.smooth) + 0.5 * self.smooth
        l = F.binary_cross_entropy_with_logits(logits.float(), t, reduction="none")
        l = l * torch.where(y > 0.5, self.wp, self.wn)
        return (l * m).sum() / m.sum().clamp(min=1)

EPS = 1e-20
def par_metrics(gt, pr, valid):
    g, p = gt[:, valid].astype(bool), pr[:, valid].astype(bool)
    tp = (g & p).sum(0); fn = (g & ~p).sum(0); tn = (~g & ~p).sum(0); fp = (~g & p).sum(0)
    rp = tp / (tp + fn + EPS); rn = tn / (tn + fp + EPS)
    mA = np.mean((rp + rn) / 2)
    prec_l = tp / (tp + fp + EPS); lf1 = np.mean(2 * prec_l * rp / (prec_l + rp + EPS))
    inter = (g & p).sum(1); union = (g | p).sum(1)
    ip = inter / (p.sum(1) + EPS); ir = inter / (g.sum(1) + EPS)
    if1 = np.mean(2 * ip * ir / (ip + ir + EPS))      # như scorer BTC: F1 từng ảnh rồi lấy trung bình
    return dict(Challenge_Avg=2 * mA * if1 / (mA + if1 + EPS), mA=mA, Instance_F1=if1, Label_F1=lf1,
                Instance_Acc=np.mean(inter / (union + EPS)), Instance_Prec=np.mean(ip), Instance_Rec=np.mean(ir))

def evaluate(frame, probs, thr=0.5):
    res = {}
    for d, idx in frame.groupby("ds").indices.items():
        gt = frame[ATTR].values[idx]
        res[d] = par_metrics(gt, probs[idx] > thr, gt.sum(0) > 0)
    r = pd.DataFrame(res).T; r.loc["MEAN"] = r.mean()
    return r

class _Scorer:
    def __init__(self, frame, probs):
        self.parts = []
        for d, idx in frame.groupby("ds").indices.items():
            gt = frame[ATTR].values[idx].astype(bool); v = gt.sum(0) > 0
            self.parts.append((gt[:, v], probs[idx][:, v], v))
    def __call__(self, thr):
        return float(np.mean([par_metrics(g, p > thr[v], np.ones(g.shape[1], bool))["Challenge_Avg"] for g, p, v in self.parts]))

def calibrate(frame, probs, grid=np.round(np.arange(0.05, 0.951, 0.025), 3), passes=2):
    # coordinate descent trên ngưỡng từng thuộc tính, mục tiêu = MEAN Challenge_Avg qua các dataset
    sc = _Scorer(frame, probs)
    thr = np.full(40, 0.5); best = sc(thr)
    for _ in range(passes):
        for a in range(40):
            for t in grid:
                old = thr[a]; thr[a] = t
                s = sc(thr)
                if s > best + 1e-6: best = s
                else: thr[a] = old
    return thr, best

## 6. Hàm train

- Head khởi tạo bias bằng logit tỷ lệ dương. Optimizer AdamW với LR giảm dần theo tầng (LLRD 0.85, head LR 1e-3, weight decay 0.05).
- Warmup 1 epoch rồi cosine, AMP fp16, clip grad 5.0, EMA (0.999). Chạy `DataParallel` khi có 2 GPU.
- Mỗi epoch đánh giá EMA trên val (ngưỡng 0.5). Lưu `best_ema.pt` (fp16) và `val_logits.npy` của epoch tốt nhất, hoặc epoch cuối với model OOF.
- `FREEZE k`: đóng băng patch embed + k khối đầu và giữ chúng ở fp16. Cần GPU có fp16, nên kiểm sai số fp16 trước khi train.
- Kết quả của một lượt train nằm ở `e2e_runs/<tên>/`, gồm `best_ema.pt`, `val_logits.npy`, `history.csv`, `meta.json`.
- Model DINOv3 / RADIO có thêm `val_logits_224x112.npy`: logit val ở độ phân giải suy luận, dùng để hiệu chỉnh ngưỡng đuôi.

In [ ]:
print("[cell] train utils", flush=True)
PAUSED = set()        # lượt train dừng giữa chừng vì hết giờ (đã lưu resume.pt)

def run_file(name, fn):
    """File `fn` của lượt train `name`: ưu tiên /kaggle/working/e2e_runs, sau đó Input (Output phiên trước gắn làm dataset)."""
    p = RUNS / name / fn
    if p.exists(): return p
    for d in input_dirs():
        for q in (d / "e2e_runs" / name / fn, d / name / fn):
            if q.exists(): return q
    return None

def gpu_gb():
    return min((torch.cuda.get_device_properties(i).total_memory / 2**30 for i in range(torch.cuda.device_count())), default=0.0)

def layer_id(n, L):
    m = re.match(r"backbone\.(?:blocks|layer|encoder\.layers)\.(\d+)\.", n)
    if m: return int(m.group(1)) + 1
    if n.startswith(("backbone.norm.", "backbone.post_layernorm.", "backbone.visual_projection.", "backbone.head.", "backbone.attn_pool.")): return L + 1
    return 0

def build_param_groups(m, sp):
    groups, vit = {}, m.arch.startswith(("dinov2", "dinov3", "clip", "siglip", "pe_core", "radio", "eva02"))
    L = len(m.backbone.blocks) if vit else 0
    for n, p_ in m.named_parameters():
        no_wd = p_.ndim <= 1 or any(k in n for k in ("cls_token", "pos_embed", "register_tokens", "reg_token", "class_embedding", "position_embedding", "probe", "latent"))
        if n.startswith(("fc.", "net.classifier.2", "net.fc.")):
            lid, lr = "head", sp["HEAD_LR"]
        elif vit:
            i = layer_id(n, L); lid, lr = f"layer{i}", sp["LR"] * sp["LLRD"] ** (L + 1 - i)
        else:
            lid, lr = "backbone", sp["LR"]
        g = groups.setdefault((lid, no_wd), {"params": [], "lr": lr, "weight_decay": 0.0 if no_wd else sp["WD"], "name": lid})
        g["params"].append(p_)
    return list(groups.values())

def prep_trainable(model, sp):
    L_vit = len(model.backbone.blocks) if model.arch.startswith(("dinov2", "dinov3", "clip", "siglip", "pe_core", "radio", "eva02")) else 0
    fz = {n_ for n_, _ in model.named_parameters() if L_vit and sp.get("FREEZE", 0) and n_.startswith("backbone.") and layer_id(n_, L_vit) <= sp["FREEZE"]}
    for n_, p_ in model.named_parameters():
        if n_ in fz:
            p_.requires_grad_(False)
            if sp.get("FROZEN_FP16") and p_.is_cuda and p_.ndim >= 2: p_.data = p_.data.half()
    if hasattr(model.backbone, "ckpt"): model.backbone.ckpt = bool(sp.get("CKPT"))
    return fz

def make_ema(model, fz, build):
    ema = build(); ema.load_state_dict(model.state_dict())
    mp = dict(model.named_parameters())
    for n_, p_ in ema.named_parameters():
        p_.requires_grad_(False)
        if n_ in fz: p_.data = mp[n_].data          # phần đóng băng dùng chung bộ nhớ với model
    return ema.to(next(model.parameters()).device).eval()

@torch.no_grad()
def predict(m, loader, tag=""):
    m.eval(); out = []
    for k, (x, *_) in enumerate(loader):
        x = x.to(dev, non_blocking=True)
        with torch.autocast(dev.type, enabled=dev.type == "cuda"):
            out.append(m(x).float().cpu())
        if tag and k % 50 == 0: print(f"  [{tag}] dự đoán lô {k}/{len(loader)}", flush=True)
    return torch.cat(out).numpy()

def clean_mask(frame):
    """CMASK: ô (ảnh, thuộc tính) bị cờ trong cmask_flags.npz -> mask 0 (bỏ khỏi loss), nhãn giữ nguyên."""
    f = run_file("e2e_flags", "cmask_flags.npz"); assert f is not None, "chưa có cmask_flags.npz (chạy stage flags trước)"
    z = np.load(f, allow_pickle=False); assert [str(a) for a in z["attr"]] == list(ATTR)
    Fl = np.unpackbits(z["flag"], axis=1, count=len(ATTR)).astype(bool)
    idx = {r: i for i, r in enumerate(z["images"].tolist())}; rows = np.array([idx.get(r, -1) for r in frame.image]); assert (rows >= 0).all()
    y = frame[ATTR].values.astype(np.float32); m = MASK[frame.ds.map(DS_ID).values].copy()
    sel = Fl[rows] & (m > 0); m[sel] = 0
    print("CMASK: ô bỏ khỏi loss", {d: int(sel[(frame.ds == d).values].sum()) for d in DS_NAMES}, f"| tổng {int(sel.sum())} ô / {int(sel.any(1).sum())} ảnh", flush=True)
    return y, m

def train_run(name, sp, train_frame, val_frame, force_last=False):
    """Train một model theo spec `sp`. Trả về thư mục run, hoặc None nếu dừng giữa chừng vì sắp hết giờ (đã lưu resume.pt)."""
    done = run_file(name, "meta.json")
    if done is not None:
        print(f"[{name}] đã train xong ({done.parent}) -> dùng lại", flush=True); return done.parent
    if PAUSED:
        print(f"[{name}] bỏ qua: phiên này đã hết giờ ({sorted(PAUSED)} dừng giữa chừng)", flush=True); return None
    out = RUNS / name; out.mkdir(parents=True, exist_ok=True)
    arch = sp["ARCH"]; H, W = sp["IMG"]; mean, std = norm_for(arch); EP = sp["EPOCHS"]; cuda = dev.type == "cuda"
    seed_all(SEED)
    ym = clean_mask(train_frame) if sp.get("CMASK") else None
    pr = pos_rate(train_frame) if ym is None else ((ym[0] * ym[1]).sum(0) / np.maximum(ym[1].sum(0), 1)).clip(1e-4, 1 - 1e-4).astype(np.float32)
    dl_tr = make_loader(train_frame, H, W, mean, std, True, sp["BATCH"], ym=ym)
    dl_va = make_loader(val_frame, H, W, mean, std, False, 256)
    model = PARNet(arch, pretrained=True, drop_path=sp["DROP_PATH"], img_h=H, img_w=W, drop=sp["HEAD_DROP"])
    with torch.no_grad(): model.head.bias.copy_(torch.as_tensor(logit(pr), dtype=torch.float32))      # đầu ra ban đầu = tỷ lệ dương
    model = model.to(dev)
    if sp.get("FP16_CHECK") and cuda:              # khối đóng băng sẽ chạy fp16: kiểm backbone không tràn số ở fp16
        xb = next(iter(dl_va))[0][:16].to(dev); model.eval()
        with torch.no_grad():
            f32 = model.backbone(xb).float()
            with torch.autocast("cuda"): f16 = model.backbone(xb).float()
        model.train(); rel = float((f16 - f32).abs().max() / f32.abs().max())
        print(f"[{name}] kiểm fp16: lệch tương đối {rel:.4f}", flush=True)
        assert np.isfinite(rel) and rel < 0.05, "backbone lệch lớn ở fp16"
    fz = prep_trainable(model, sp)
    ema = make_ema(model, fz, lambda: PARNet(arch, pretrained=False, img_h=H, img_w=W))
    opt = torch.optim.AdamW(build_param_groups(model, sp))
    steps_ep = len(dl_tr); total = steps_ep * EP; warm = int(steps_ep * 1.0)
    base_lrs = [g["lr"] for g in opt.param_groups]
    def set_lr(step):
        f = step / max(1, warm) if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
        for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * f
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    crit = MaskedLoss(pr, 0.05).to(dev)
    start_ep, step, best, hist, ep_time = 0, 0, -1.0, [], sp["EST_EP_MIN"] * 60.0
    rs = run_file(name, "resume.pt")
    if rs is not None:                             # tiếp tục lượt train dừng ở phiên trước
        ck = torch.load(rs, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"], strict=False); ema.load_state_dict(ck["ema"], strict=False)
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start_ep, step, best, hist, ep_time = ck["next_ep"], ck["step"], ck["best"], ck["hist"], ck["ep_time"]
        for fn in ("best_ema.pt", "val_logits.npy"):
            if rs.parent != out and (rs.parent / fn).exists(): shutil.copy2(rs.parent / fn, out / fn)
        del ck; seed_all(SEED + 1000 * start_ep)
        print(f"[{name}] tiếp tục từ epoch {start_ep + 1}/{EP} ({rs})", flush=True)
    msd, esd = model.state_dict(), ema.state_dict()
    def ema_update(step):
        d = min(0.999, (1 + step) / (10 + step))
        for k, v in esd.items():
            if k in fz: continue
            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else: v.copy_(msd[k])
    net = nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
    print(f"[{name}] {arch} {H}x{W} | train {len(train_frame)} ảnh, {steps_ep} bước/epoch x {EP} epoch | val {len(val_frame)} "
          f"| đóng băng {len(fz)} tensor | {torch.cuda.device_count()} GPU", flush=True)
    for ep in range(start_ep, EP):
        if time.time() - T_START + 1.1 * ep_time > TIME_LIMIT_H * 3600:
            if ep == 0:                            # chưa train bước nào: không cần resume.pt, phiên sau train từ đầu
                print(f"[{name}] không đủ giờ cho epoch đầu -> để phiên sau", flush=True)
                PAUSED.add(name); net = model = ema = opt = msd = esd = dl_tr = dl_va = None; gc.collect(); torch.cuda.empty_cache(); return None
            torch.save(dict(model={k: v for k, v in model.state_dict().items() if k not in fz}, ema={k: v for k, v in ema.state_dict().items() if k not in fz},
                            opt=opt.state_dict(), scaler=scaler.state_dict(), next_ep=ep, step=step, best=best, hist=hist, ep_time=ep_time), out / "resume.pt")
            print(f"[{name}] sắp hết {TIME_LIMIT_H} h -> lưu resume.pt, dừng trước epoch {ep + 1}. Gắn Output phiên này làm Input rồi chạy lại stage.", flush=True)
            PAUSED.add(name); net = model = ema = opt = msd = esd = dl_tr = dl_va = None; gc.collect(); torch.cuda.empty_cache(); return None
        t_ep = time.time(); model.train(); tot, nb, bad = 0.0, 0, 0
        for it, (x, y, m, _) in enumerate(dl_tr):
            x, y, m = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True), m.to(dev, non_blocking=True); set_lr(step)
            with torch.autocast(dev.type, enabled=cuda):
                zl = net(x).float(); loss = crit(zl, y, m)
            if not torch.isfinite(loss):
                bad += 1; assert bad < 50, f"[{name}] loss không hữu hạn 50 lần liên tiếp"
                opt.zero_grad(set_to_none=True); continue
            bad = 0
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0); scaler.step(opt); scaler.update()
            with torch.no_grad(): ema_update(step)
            step += 1; tot += float(loss); nb += 1
            if it % 50 == 0:
                el = time.time() - t_ep
                print(f"[{name}] ep {ep + 1}/{EP} it {it}/{steps_ep} loss {tot / nb:.4f} | {nb * sp['BATCH'] / max(el, 1e-6):.0f} img/s "
                      f"| còn ~{el / (it + 1) * (steps_ep - it - 1) / 60:.0f} phút", flush=True)
        z_va = predict(ema, dl_va); r = evaluate(val_frame, sigm(z_va)); sel = float(r.loc["MEAN", "Challenge_Avg"])
        row = dict(epoch=ep + 1, loss=tot / max(nb, 1), val_raw_avg=sel, min=(time.time() - t_ep) / 60,
                   **{f"{d}_avg": float(r.loc[d, "Challenge_Avg"]) for d in r.index if d != "MEAN"})
        hist.append(row)
        if sel > best or force_last:
            best = sel
            torch.save({k: (v.detach().cpu().half() if v.dtype == torch.float32 else v.detach().cpu()) for k, v in ema.state_dict().items()}, out / "best_ema.pt")
            np.save(out / "val_logits.npy", z_va)
        ep_time = time.time() - t_ep
        print(f"[{name}] epoch {ep + 1}/{EP}: loss {row['loss']:.4f} | val {sel:.4f} (tốt nhất {best:.4f}) | {ep_time / 60:.1f} phút", flush=True)
    if sp.get("EXTRA_VAL") and arch.startswith(("dinov3", "radio")):     # logit val ở 224x112 (độ phân giải suy luận) cho ngưỡng đuôi
        ema.load_state_dict({k: v.float() for k, v in torch.load(out / "best_ema.pt", map_location="cpu").items()})
        np.save(out / "val_logits_224x112.npy", predict(ema, make_loader(val_frame, 224, 112, mean, std, False, 256), f"{name} 224x112"))
    pd.DataFrame(hist).to_csv(out / "history.csv", index=False)
    json.dump(dict(name=name, arch=arch, parts=False, wb=False, qdec=False, cmid=[], chist=False, img_h=H, img_w=W, mean=list(mean), std=list(std),
                   **{k: v for k, v in sp.items() if k != "ARCH"}, full=False, select="last" if force_last else "best val"),
              open(out / "meta.json", "w"), indent=1)                      # meta.json ghi sau cùng = dấu hiệu lượt train đã xong
    (out / "resume.pt").unlink(missing_ok=True)
    print(pd.DataFrame(hist).round(4).to_string(index=False), flush=True)
    net = model = ema = opt = msd = esd = dl_tr = dl_va = None; gc.collect(); torch.cuda.empty_cache()
    return out

def teacher_logits(tdir, frame, tag=""):
    meta = json.load(open(tdir / "meta.json"))
    m = PARNet(meta["arch"], pretrained=False, img_h=meta["img_h"], img_w=meta["img_w"])
    m.load_state_dict({k: v.float() for k, v in torch.load(tdir / "best_ema.pt", map_location="cpu").items()})
    z = predict(m.to(dev), make_loader(frame, meta["img_h"], meta["img_w"], tuple(meta["mean"]), tuple(meta["std"]), False, 256), tag)
    del m; gc.collect(); torch.cuda.empty_cache()
    return z

## 7. Cấu hình các lượt train

Chung cho mọi lượt: 8 epoch, LR 3e-5, LLRD 0.85, drop path 0.15, head LR 1e-3, batch 64, dropout head 0.2.
- `EST_EP_MIN`: ước lượng phút / epoch (T4 x2; f8c trên A100). Chỉ dùng cho mốc dừng ở epoch đầu của phiên.

In [ ]:
print("[cell] run specs", flush=True)
BASE = dict(EPOCHS=8, LR=3e-5, LLRD=0.85, DROP_PATH=0.15, HEAD_LR=1e-3, WD=0.05, BATCH=64, HEAD_DROP=0.2)
HPLUS = dict(FROZEN_FP16=True, CKPT=False, FP16_CHECK=True, EXTRA_VAL=True)
RUN_SPECS = {
    "oof":       dict(BASE, ARCH="dinov3_vitl16", IMG=(256, 128), EST_EP_MIN=22),                         # 2 fold OOF cho CMASK
    "e2e_l2c":   dict(BASE, ARCH="dinov2_vitl14_reg", IMG=(252, 126), CMASK=True, EST_EP_MIN=35),          # weights_1
    "e2e_g11":   dict(BASE, ARCH="dinov3_vith16p", IMG=(256, 128), FREEZE=20, **HPLUS, EST_EP_MIN=52),     # weights_2 (đuôi)
    "e2e_radio": dict(BASE, ARCH="radio_v4h", IMG=(256, 128), FREEZE=12, **HPLUS, EST_EP_MIN=48),          # weights_3 (đuôi)
    "e2e_f8c":   dict(BASE, ARCH="dinov3_vith16p", IMG=(256, 128), FREEZE=8, **HPLUS, CMASK=True, EST_EP_MIN=40),   # weights_0, cần GPU >= 39 GB
}
for k, v in RUN_SPECS.items(): print(k, v)

## 8. CMASK, bước 1: 2 fold OOF (stage `oof0`, `oof1`)

- Chia train thành nhóm:
  - Market: mã người.
  - PA100k: khối 20 chỉ số.
  - PETA: chuỗi ảnh liền nhau, chỉ số cách ≤ 10 và nhãn lệch ≤ 2 ô so với ảnh trước.
- Fold của mỗi nhóm = crc32(tên nhóm) mod 2.
- Fold f: train DINOv3-L trên các nhóm còn lại 8 epoch (lấy epoch cuối), rồi dự đoán ảnh của fold f.
- Ngưỡng hiệu chỉnh trên val được lưu vào `oof_summary.json`.

In [ ]:
print("[cell] oof folds", flush=True)
def oof_groups(frame):
    g = {}
    for im, d in zip(frame.image, frame.ds):
        f = im.rsplit("/", 1)[-1]
        if d == "Market1501": g[im] = "M" + f[:4]
        elif d == "PA100k": g[im] = "A%d" % (int(re.sub(r"\D", "", f.split(".")[0]) or 0) // 20)
    pe = frame[frame.ds == "PETA"].copy()
    pe["n"] = pe.image.str.extract(r"(\d+)\.\w+$")[0].astype(int).values; pe = pe.sort_values("n")
    Yp, n, ims = pe[ATTR].values, pe.n.values, pe.image.values; gid = None
    for k in range(len(pe)):
        if k == 0 or n[k] - n[k - 1] > 10 or np.abs(Yp[k] - Yp[k - 1]).sum() > 2: gid = n[k]
        g[ims[k]] = f"P{gid}"
    return g

GRP = oof_groups(TR)
FO = np.array([zlib.crc32(GRP[i].encode()) % 2 for i in TR.image])
print("ảnh mỗi fold:", np.bincount(FO).tolist())
for f in (0, 1):
    name = f"e2e_oof_f{f}"
    if f"oof{f}" not in STAGES: print(f"[{name}] không có trong STAGES -> bỏ qua"); continue
    if run_file(name, "oof_summary.json") is not None: print(f"[{name}] đã có kết quả OOF -> bỏ qua"); continue
    TR_FIT, TR_OOF = TR[FO != f].reset_index(drop=True), TR[FO == f].reset_index(drop=True)
    assert not (set(TR_FIT.image.map(GRP)) & set(TR_OOF.image.map(GRP))), "nhóm nằm ở cả 2 phía"
    print(f"[{name}] train {len(TR_FIT)} {TR_FIT.ds.value_counts().to_dict()} | dự đoán {len(TR_OOF)}", flush=True)
    d = train_run(name, RUN_SPECS["oof"], TR_FIT, VA, force_last=True)
    if d is None: continue
    od = RUNS / name; od.mkdir(parents=True, exist_ok=True)
    np.save(od / "oof_logits.npy", teacher_logits(d, TR_OOF, f"{name} OOF").astype(np.float32))
    TR_OOF[["image"]].to_csv(od / "oof_images.csv", index=False); VA[["image"]].to_csv(od / "val_images.csv", index=False)
    thr = calibrate(VA, sigm(np.load(run_file(name, "val_logits.npy"))))[0]
    json.dump({"thr_val": thr.tolist()}, open(od / "oof_summary.json", "w"))
    print(f"[{name}] OOF xong", flush=True)

## 9. CMASK, bước 2: gắn cờ ô nhãn nghi sai (stage `flags`)

Một ô (ảnh, thuộc tính) bị gắn cờ khi:
- **0 → 1**: nhãn 0, margin = logit OOF − logit(ngưỡng val) > 2, và xác suất Platt ≥ 0.9.
- **1 → 0**: nhãn 1, margin < −2, xác suất Platt ≤ 0.1, và thuộc tính không nằm trong nhóm giữ dương (balo, túi, kính thường, kính râm, mũ).

Platt hiệu chỉnh trên val theo từng cặp (dataset, thuộc tính), cần ≥ 20 dương và ≥ 20 âm, không đủ thì dùng hệ số chung. Không xét ô của thuộc tính mà dataset không gán nhãn.

Bản gốc có 19 917 ô (11 122 ô 0→1, 8 795 ô 1→0) trên 14 585 ảnh. Train lại sẽ ra số hơi khác.

In [ ]:
print("[cell] cmask flags", flush=True)
MARGIN_T, P_HI, P_LO = 2.0, 0.9, 0.1
KEEP_POS = ["Accessory-Backpack", "Accessory-Bag", "Accessory-Glasses-Normal", "Accessory-Glasses-Sun", "Accessory-Hat"]

def platt(z, y, it=50, l2=1e-3):
    a_, b_ = 1.0, 0.0
    for _ in range(it):
        p = 1 / (1 + np.exp(-np.clip(a_ * z + b_, -30, 30))); w = p * (1 - p)
        g = np.array([np.sum((p - y) * z) + l2 * (a_ - 1), np.sum(p - y)])
        Hm = np.array([[np.sum(w * z * z) + l2, np.sum(w * z)], [np.sum(w * z), np.sum(w) + 1e-9]])
        st = np.linalg.solve(Hm, g); a_, b_ = a_ - st[0], b_ - st[1]
        if np.abs(st).max() < 1e-6: break
    return a_, b_

def calib_tables(name):
    # Platt (dataset, thuộc tính) trên val của lượt OOF; trả về hàm (z, dataset) -> p
    zv = np.load(run_file(name, "val_logits.npy")).astype(np.float64); ims = pd.read_csv(run_file(name, "val_images.csv")).image.values
    vidx = {im: i for i, im in enumerate(VA.image)}; vds = VA.ds.values
    rows = np.array([vidx[i] for i in ims]); yv = VA[ATTR].values[rows] > 0.5; dv = vds[rows]
    glob = [platt(zv[:, j], yv[:, j].astype(float)) if 0 < yv[:, j].sum() < len(rows) else (1.0, 0.0) for j in range(40)]
    tab = {}
    for d in np.unique(vds):
        s = dv == d
        for j in range(40):
            npos = yv[s, j].sum()
            tab[d, j] = platt(zv[s, j], yv[s, j].astype(float)) if npos >= 20 and s.sum() - npos >= 20 else glob[j]
    A_ = {d: np.array([tab[d, j][0] for j in range(40)]) for d in np.unique(vds)}; B_ = {d: np.array([tab[d, j][1] for j in range(40)]) for d in np.unique(vds)}
    def f(z, dset):
        p = np.zeros_like(z)
        for d in np.unique(dset):
            r = dset == d; p[r] = 1 / (1 + np.exp(-np.clip(z[r] * A_[d][None] + B_[d][None], -30, 30)))
        return p
    return f

if "flags" in STAGES and run_file("e2e_flags", "cmask_flags.npz") is None and not PAUSED:
    Y = TR[ATTR].values.astype(np.float64); ds = TR.ds.values
    mask = np.ones((len(TR), 40), bool)                       # thuộc tính chưa từng dương ở dataset đó -> không đụng
    for d in np.unique(ds):
        i = ds == d; mask[i] &= (Y[i].sum(0) > 0)[None]
        if d == "PETA":
            for c in ("UpperBody-Color-Other", "LowerBody-Color-Other"): mask[i, ATTR.index(c)] = False
    pos_ok = np.ones(40, bool); pos_ok[[ATTR.index(k) for k in KEEP_POS]] = False
    idx = {im: i for i, im in enumerate(TR.image)}
    m = np.full((len(TR), 40), np.nan); pr = np.full((len(TR), 40), np.nan)
    for f in (0, 1):
        name = f"e2e_oof_f{f}"; assert run_file(name, "oof_summary.json") is not None, f"thiếu kết quả {name} (stage oof{f})"
        z = np.load(run_file(name, "oof_logits.npy")).astype(np.float64); ims = pd.read_csv(run_file(name, "oof_images.csv")).image.values
        thr = np.clip(np.array(json.load(open(run_file(name, "oof_summary.json")))["thr_val"]), 1e-4, 1 - 1e-4)
        rows = np.array([idx[i] for i in ims]); assert np.isnan(m[rows]).all(), "ảnh trùng giữa 2 fold"
        m[rows] = z - np.log(thr / (1 - thr))[None]; pr[rows] = calib_tables(name)(z, ds[rows])
    assert not np.isnan(m).any(), "OOF chưa phủ hết train"
    up = (m > MARGIN_T) & (Y < 0.5) & mask & (pr >= P_HI)
    down = (m < -MARGIN_T) & (Y > 0.5) & mask & pos_ok[None] & (pr <= P_LO)
    fl = up | down
    print(f"ô gắn cờ: {int(fl.sum())} (0->1 {int(up.sum())}, 1->0 {int(down.sum())}) trên {int(fl.any(1).sum())} ảnh "
          f"| bản gốc 19917 (11122 / 8795) trên 14585 ảnh")
    for d in DS_NAMES: print(f"  {d}: 0->1 {int(up[ds == d].sum())}, 1->0 {int(down[ds == d].sum())}")
    (RUNS / "e2e_flags").mkdir(parents=True, exist_ok=True)
    np.savez_compressed(RUNS / "e2e_flags/cmask_flags.npz", images=np.array(TR.image.tolist(), dtype="U"),
                        attr=np.array(ATTR, dtype="U"), flag=np.packbits(fl, axis=1))
else:
    print("flags:", run_file("e2e_flags", "cmask_flags.npz") or ("bỏ qua, OOF dừng giữa chừng" if PAUSED else "không có trong STAGES"))

## 10. Train 4 model của gói nộp (stage `l2c`, `g11`, `radio`, `f8c`)

In [ ]:
print("[cell] train l2c", flush=True)
if "l2c" in STAGES: train_run("e2e_l2c", RUN_SPECS["e2e_l2c"], TR, VA)
else: print("l2c không có trong STAGES:", run_file("e2e_l2c", "meta.json"))

In [ ]:
print("[cell] train g11", flush=True)
if "g11" in STAGES: train_run("e2e_g11", RUN_SPECS["e2e_g11"], TR, VA)
else: print("g11 không có trong STAGES:", run_file("e2e_g11", "meta.json"))

In [ ]:
print("[cell] train radio", flush=True)
if "radio" in STAGES: train_run("e2e_radio", RUN_SPECS["e2e_radio"], TR, VA)
else: print("radio không có trong STAGES:", run_file("e2e_radio", "meta.json"))

In [ ]:
print("[cell] train f8c", flush=True)
if "f8c" not in STAGES:
    print("f8c không có trong STAGES:", run_file("e2e_f8c", "meta.json"))
elif run_file("e2e_f8c", "meta.json") is None and gpu_gb() < 39 and not os.environ.get("UPAR_FORCE_F8C"):
    print(f"BỎ QUA f8c: ViT-H+ đóng băng 8 khối cần GPU >= 39 GB, máy này {gpu_gb():.0f} GB -> chạy stage f8c trên A100", flush=True)
else:
    train_run("e2e_f8c", RUN_SPECS["e2e_f8c"], TR, VA)

## 11. Đóng gói (stage `pack`)

### 11a. Weights

- Nạp `best_ema.pt` của từng model.
- L2c train ở 252x126 nên nội suy bicubic `pos_embed` từ lưới 18x9 sang 16x8 (224x112).
- Nén w8: ma trận ≥ 4096 phần tử thành int8 đối xứng theo kênh ra với scale fp16, phần còn lại fp16. `run.py` giải nén khi nạp.

In [ ]:
print("[cell] pack weights", flush=True)
PACK = [("e2e_f8c", 0.24, False), ("e2e_l2c", 0.16, False), ("e2e_g11", 0.3, True), ("e2e_radio", 0.3, True)]   # (run, w, optional = chỉ đuôi)

def deq(sd):
    if any(k.endswith("::w8scale") for k in sd):
        sd = {k: v.float() * sd[k + "::w8scale"].float() if k + "::w8scale" in sd else v for k, v in sd.items() if not k.endswith("::w8scale")}
    return {k: v.float() if v.is_floating_point() else v for k, v in sd.items()}

def w8(sd):
    q = {}
    for k, v in sd.items():
        v = v.half() if v.dtype == torch.float32 else v
        if v.is_floating_point() and v.ndim >= 2 and v.numel() >= 4096:
            f = v.float(); sc = f.abs().amax(dim=tuple(range(1, f.ndim)), keepdim=True).clamp_min(1e-8) / 127.0
            q[k] = torch.round(f / sc).clamp(-127, 127).to(torch.int8); q[k + "::w8scale"] = sc.half()
        else: q[k] = v
    return q

def resize_pos_embed(sd, th, tw, key="backbone.pos_embed"):
    pe = sd[key]; C = pe.shape[-1]; n0 = pe.shape[1] - 1
    gh, gw = int(round((n0 * 2) ** .5)), int(round((n0 / 2) ** .5)); assert gh * gw == n0, (gh, gw, n0)
    g = F.interpolate(pe[0, 1:].reshape(1, gh, gw, C).permute(0, 3, 1, 2).float(), size=(th, tw), mode="bicubic", align_corners=False, antialias=True)
    sd = dict(sd); sd[key] = torch.cat([pe[:, :1].float(), g.permute(0, 2, 3, 1).reshape(1, -1, C)], 1)
    print(f"  pos_embed {gh}x{gw} -> {th}x{tw}")
    return sd

MODELS = []
DO_PACK = "pack" in STAGES and not PAUSED          # có lượt train dừng giữa chừng thì chưa đóng gói
if "pack" in STAGES and PAUSED: print("bỏ qua pack: còn lượt train dừng giữa chừng", sorted(PAUSED))
if DO_PACK:
    miss = [n for n, *_ in PACK if run_file(n, "meta.json") is None]
    assert not miss, f"thiếu lượt train {miss}: chạy các stage đó trước, hoặc gắn Output của chúng làm Input"
    for i, (name, w, optional) in enumerate(PACK):
        meta = json.load(open(run_file(name, "meta.json")))
        sd = deq(torch.load(run_file(name, "best_ema.pt"), map_location="cpu", weights_only=True))
        rz = meta["arch"].startswith("dinov2") and (meta["img_h"], meta["img_w"]) != (224, 112)
        if rz: sd = resize_pos_embed(sd, 224 // 14, 112 // 14)
        torch.save(w8(sd), PKG / "assets" / f"weights_{i}.pt")
        ent = dict(arch=meta["arch"], parts=False, qdec=False, wb=False, cmid=[], chist=False, img_h=224, img_w=112,
                   mean=list(meta["mean"]), std=list(meta["std"]),
                   source=f"{name} (train {meta['img_h']}x{meta['img_w']}, FREEZE {meta.get('FREEZE', 0)}, {'CMASK' if meta.get('CMASK') else 'nhãn gốc'}) | w8"
                          + (", pos_embed 224x112" if rz else ""),
                   file=f"weights_{i}.pt", w=w)
        if optional: ent["optional"] = True
        MODELS.append(ent)
        print(f"weights_{i}.pt <- {run_file(name, 'best_ema.pt')} ({(PKG / 'assets' / f'weights_{i}.pt').stat().st_size / 1e6:.0f} MB) w {w}"
              + (" (chỉ đuôi)" if optional else ""), flush=True)
        del sd; gc.collect()

### 11b. `run.py` của gói nộp

Nguyên văn `run.py` (`run_split`) trong zip bản gốc. Chương trình chấm của BTC import file này rồi gọi `predict_batch(samples)`.
- **Chia cứng theo vị trí ảnh** (`split.at` = 28095):
  - Ảnh trước mốc (MEVID) chạy model không `optional` (F8c + L2c) với `thresholds_solo`.
  - Ảnh từ mốc trở đi (Synth) chạy cả 4 model + lật ngang (`tail_flip`) với `thresholds`.
- **Chốt chặn thời gian** (5 phút trên A10): tự đo tốc độ. Nếu dự báo vượt ngân sách thì bỏ model phụ, rồi hạ độ phân giải.
- Bản gốc dùng xuống dòng CRLF. Khi ghi vào zip, notebook đổi lại CRLF để byte trùng bản gốc.

In [ ]:
%%module run
# UPAR 2027 Track 1 submission (v2). Cấu trúc: run.py, metadata.yaml, model.py, assets/{config.json, weights_*.pt}
# Chốt chặn thời gian: run.py tự đo tốc độ sau mỗi batch. Nếu dự báo tổng thời gian vượt ngân sách, các ảnh còn lại chạy ở độ
# phân giải thấp hơn (cùng weights, nội suy pos_embed như research/make_lowres_zip.py; trên val 224x112 mất ~0.001, 196x98 ~0.005).
# Server đủ nhanh thì luôn ở mức 0 = đúng như bản không có chốt chặn. Log chỉ dùng ASCII để không lỗi mã hoá stderr.
# Model phụ ("optional": true trong config, cần "thresholds_solo"): nếu dự báo vượt ngân sách thì bỏ model phụ trước (các ảnh còn lại
# chỉ chạy model chính với ngưỡng riêng của nó, độ phân giải giữ nguyên); chỉ hạ độ phân giải khi đã bỏ model phụ và time_guard bật.
# v17x (14/10): model có "head_only": true chỉ chạy khi model phụ TẮT (phần đầu, ngưỡng solo); khi model phụ BẬT (đuôi) nó bị bỏ qua, nên đuôi chạy
# đúng bộ (model chính không head_only + model phụ) với ngưỡng 3 model. Dự báo thời gian của đuôi trừ chi phí model head_only (đo riêng).
# run_split (18/10): config "split": {"at": N} chia cứng theo vị trí ảnh trong luồng test (đếm như log "new group ... at image #"; test:
# MEVID #0-28094, Synth từ #28095). Ảnh vị trí < N = ĐẦU: model không "optional", ngưỡng thresholds_solo, độ phân giải theo time guard.
# Ảnh vị trí >= N = ĐUÔI: model không "head_only" + model phụ, ngưỡng thresholds, mức tail_tier. Lô vắt qua mốc được tách làm 2.
# Model phụ không bao giờ chạy ở đầu; ở đuôi chỉ bị bỏ (đuôi dùng bộ đầu) nếu dự báo vượt hard_s. Không có "split" = y như run_v17x.
# 18/10 tối: lần chấm split_c1 bỏ oan model phụ — lô vắt mốc chỉ có 1 ảnh đuôi + lần chạy nguội đầu tiên của model phụ đo ra 109 ms/ảnh
# (dự báo 364 s, thực dư ~50 s). Nay ở chế độ split bỏ đo tốc độ của lô < opt_min_n ảnh (mặc định 16) và lô đủ cỡ đầu tiên có model phụ.
from __future__ import annotations
import os, sys, json, time
import numpy as np, torch
import torch.nn.functional as F
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
HERE = os.path.dirname(os.path.abspath(__file__))
ASSETS = os.path.join(HERE, "assets")
sys.path.insert(0, HERE)
from model import PARNet, preprocess, crop_frac

BATCH_SIZE = 64
T0 = time.time()
# budget_s tính từ lúc import run.py; n_expected = số ảnh test (log ingestion: 29 248); costs = thời gian tương đối từng mức
# (đo 04/10 bằng torch 2.4.1, 6 luồng, 300 ảnh val: 252x126 0.8763 | 224x112 0.8732 | 196x98 0.8704 | 168x84 0.8676 | 140x70 0.8646)
GUARD = dict(budget_s=250.0, n_expected=29248, scales=[1.0, 0.889, 0.778, 0.667, 0.556], costs=[1.0, 0.86, 0.65, 0.48, 0.37],   # 08/10: chấm GPU, giới hạn 300 s
             hard_s=280.0, tail_n=3000, tail_tier=1)   # v17e (14/10): tail_n ảnh cuối chạy ở mức tier_tail (192x96), như lượt v17 0.7992 (Synth 0.8529)   # 11/10: model phụ đã bật ở tail_n ảnh cuối (gồm Synth = 50 % điểm) chỉ bị tắt lại khi dự báo vượt hard_s
_S = {}

def _log(msg):
    try: print(f"[run.py] {time.time() - T0:5.0f}s | {msg}", file=sys.stderr, flush=True)
    except Exception: pass

def _read(path, n=1 << 16):
    try:
        with open(path) as fh: return fh.read(n)
    except Exception:
        return ""

def _diag():
    # Thông tin máy chấm, chỉ để hiểu vì sao tốc độ dao động; không ảnh hưởng dự đoán
    info = {"python": sys.version.split()[0], "cpu_count": os.cpu_count(), "threads": torch.get_num_threads(),
            "cgroup_cpu": (_read("/sys/fs/cgroup/cpu.max") or _read("/sys/fs/cgroup/cpu/cpu.cfs_quota_us")).strip()}
    try: info["affinity"] = len(os.sched_getaffinity(0))
    except Exception: pass
    cpu = _read("/proc/cpuinfo").splitlines()
    info["cpu"] = next((l.split(":", 1)[1].strip() for l in cpu if l.startswith("model name")), "?")
    flags = next((l.split(":", 1)[1].split() for l in cpu if l.startswith("flags")), [])
    info["flags"] = [f for f in ("avx2", "avx512f", "avx512_vnni", "avx512_bf16", "amx_bf16", "amx_int8") if f in flags]
    mem = next((l.split()[1] for l in _read("/proc/meminfo").splitlines() if l.startswith("MemTotal")), "0")
    info["mem_gb"] = round(int(mem) / 2**20, 1)
    _log("diag " + json.dumps(info))

def load_model() -> None:
    cfg = json.load(open(os.path.join(ASSETS, "config.json")))
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if dev.type == "cpu": torch.set_num_threads(max(1, os.cpu_count() or 1))
    models = []
    for sp in cfg["models"]:
        mk = lambda: PARNet(sp["arch"], pretrained=False, img_h=sp["img_h"], img_w=sp["img_w"], parts=sp.get("parts", False), qdec=sp.get("qdec", False), cmid=tuple(sp.get("cmid", ())), chist=bool(sp.get("chist", False)))
        sd = torch.load(os.path.join(ASSETS, sp["file"]), map_location="cpu", weights_only=True)
        if any(k.endswith("::w8scale") for k in sd):              # weights int8 theo kênh (bản _w8): giải nén
            sd = {k: v.float() * sd[k + "::w8scale"].float() if k + "::w8scale" in sd else v for k, v in sd.items() if not k.endswith("::w8scale")}
        sd = {k: v.float() if v.is_floating_point() else v for k, v in sd.items()}
        m = None
        try:                                                # v17x: dựng trên thiết bị meta (bỏ khởi tạo ngẫu nhiên ~1 s / 300M tham số) rồi gán weights
            with torch.device("meta"): m = mk()
            m.load_state_dict(sd, assign=True)
            if any(t.is_meta for t in list(m.parameters()) + list(m.buffers())): raise RuntimeError("tensor meta chưa được gán")
        except Exception as e:
            _log(f"meta init off for {sp['arch']}: {e!r}"); m = mk(); m.load_state_dict(sd)
        m.to(dev).eval()
        if dev.type == "cpu" and cfg.get("cpu_int8", False):
            skip = tuple(cfg.get("int8_skip", []))          # vd ["fc2"]: giữ fp32 các Linear có tên kết thúc bằng các chuỗi này
            spec = {n: torch.ao.quantization.default_dynamic_qconfig for n, mod in m.named_modules()
                    if isinstance(mod, torch.nn.Linear) and not (skip and n.endswith(skip))}
            m = torch.ao.quantization.quantize_dynamic(m, spec, dtype=torch.qint8)
        sp = dict(sp)
        sp["_w"] = torch.as_tensor(sp["w"], dtype=torch.float32, device=dev)          # số hoặc vector 40 (trọng số theo thuộc tính, 09/10)
        sp["_crop"] = tuple(sp.get("crop") or ())                                       # model chuyên vùng đầu: cắt theo tỷ lệ trước khi resize
        sp["_mean"] = torch.tensor(sp["mean"], dtype=torch.float32, device=dev).view(1, 3, 1, 1)      # chuẩn hoá cả lô (v6)
        sp["_std"] = torch.tensor(sp["std"], dtype=torch.float32, device=dev).view(1, 3, 1, 1)
        if sp["arch"].startswith("dinov2"):                 # DINOv2 đổi độ phân giải bằng nội suy pos_embed
            sp.update(pe0=m.backbone.pos_embed.detach().clone(), grid0=tuple(m.backbone.grid), pe={})
        elif sp["arch"].startswith(("dinov3", "radio")):    # DINOv3 (RoPE) / C-RADIO (CPE): chỉ cần đổi kích thước ảnh
            sp.update(pe0=None, grid0=tuple(m.backbone.grid), pe={})
        models.append((m, sp))
    thr = np.clip(np.array(cfg["thresholds"], dtype=np.float64), 1e-4, 1 - 1e-4)
    has_opt = any(sp.get("optional") for _, sp in models) and cfg.get("thresholds_solo") is not None
    thr_solo = np.clip(np.array(cfg["thresholds_solo"] if has_opt else cfg["thresholds"], dtype=np.float64), 1e-4, 1 - 1e-4)
    g = dict(GUARD, **(cfg.get("time_guard") or {}))
    for key, env in (("budget_s", "UPAR_TIME_BUDGET"), ("n_expected", "UPAR_N_EXPECTED")):
        if os.environ.get(env): g[key] = float(os.environ[env])
    split_at = (cfg.get("split") or {}).get("at")
    if os.environ.get("UPAR_SPLIT_AT"): split_at = int(os.environ["UPAR_SPLIT_AT"])
    split_at = int(split_at) if (split_at is not None and has_opt) else None
    if cfg.get("time_guard") is False or os.environ.get("UPAR_GUARD_OFF") or not all("pe0" in sp for _, sp in models):
        g.update(scales=[1.0], costs=[1.0])
    g.update(k=0, n_done=0, last_t=None, last_n=0, last_k=0, calls=0, rates=[], n_tier=[0] * len(g["scales"]), groups={}, next_log=2880,
             has_opt=has_opt, opt_on=split_at is None, opt_rates=[], last_o=0.0, n_opt=0, ho_rates=[], last_h=0.0,
             shadow_on=True, sh_rates=[], last_s=0.0, t_io=0.0, t_prep=0.0, t_fwd=0.0, t_shadow=0.0, n_shadow=0,
             split_at=split_at, tail_ok=True, tail_seen=False, part_groups={}, idx_mismatch=0, opt_full=0, last_valid=True)
    print(f"[run.py] torch {torch.__version__} | device {dev}"
          + (f" ({torch.cuda.get_device_name(0)})" if dev.type == "cuda" else "")
          + f" | cpu threads {torch.get_num_threads()} | models "
          + ", ".join(f"{sp['arch']} {sp['img_h']}x{sp['img_w']} w=" + (f"{sp['w']:.2f}" if not isinstance(sp['w'], list) else f"vec{max(sp['w']):.2f}")
                      + (" crop" if sp.get("crop") else "") + (" optional" if sp.get("optional") else "") + (" head_only" if sp.get("head_only") else "") for _, sp in models)
          + f" | tta {cfg.get('tta_flip', False)} (tail {cfg.get('tail_flip', False)}) | int8 {dev.type == 'cpu' and cfg.get('cpu_int8', False)}"
          + f" | guard {g['budget_s']:.0f}s/{int(g['n_expected'])} img, {len(g['scales'])} tiers"
          + (f" | split at image #{split_at} (tail tier {g.get('tail_tier', 0)}, hard {g.get('hard_s', g['budget_s']):.0f}s)" if split_at is not None else ""),
          file=sys.stderr, flush=True)
    _S.update(models=models, dev=dev, cfg=cfg, names=cfg["attributes"], guard=g,
              off=torch.tensor(np.log(thr / (1 - thr)), dtype=torch.float32, device=dev),
              off_solo=torch.tensor(np.log(thr_solo / (1 - thr_solo)), dtype=torch.float32, device=dev))
    try: _diag()
    except Exception as e: _log(f"diag error {e!r}")

def _open(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

def _size(sp, k):
    if k == 0: return sp["img_h"], sp["img_w"]
    s = _S["guard"]["scales"][k]; q = 16 if sp["arch"].startswith(("dinov3", "radio")) else 14
    return max(q, round(sp["img_h"] * s / q) * q), max(q, round(sp["img_w"] * s / q) * q)

def _prep(img, sp, h, w):
    if img is None: return torch.zeros(3, h, w)
    return preprocess(img, h, w, tuple(sp["mean"]), tuple(sp["std"]), wb=sp.get("wb", False))

def _load_resize(path, keys):
    # v6: luồng CPU chỉ giải mã + (cắt) + resize PIL (BILINEAR như preprocess() lúc train); đổi sang tensor + chuẩn hoá làm cả lô trên GPU
    im = _open(path)
    if im is None: return None
    return {key: np.asarray(crop_frac(im, key[2]).resize((key[1], key[0]), resample=2)) for key in keys}

def _grayworld_b(x):
    m = x.mean((2, 3)); g = (m.mean(1, keepdim=True) / m.clamp(min=1e-3)).clamp(0.6, 1.6)
    return (x * g[:, :, None, None]).clamp(0, 1)

def _set_tier(k):
    # Đổi độ phân giải: nội suy bicubic pos_embed đã train (lưới grid0) sang lưới của mức k
    for m, sp in _S["models"]:
        q = 16 if sp["arch"].startswith(("dinov3", "radio")) else 14
        h, w = _size(sp, k); grid = (h // q, w // q)
        if sp["pe0"] is None: m.backbone.grid = grid; continue
        if grid not in sp["pe"]:
            pe0, (gh, gw) = sp["pe0"], sp["grid0"]
            if grid == (gh, gw): sp["pe"][grid] = pe0
            else:
                C = pe0.shape[-1]
                g = pe0[0, 1:].float().reshape(1, gh, gw, C).permute(0, 3, 1, 2)
                g = F.interpolate(g, size=grid, mode="bicubic", align_corners=False, antialias=True)
                sp["pe"][grid] = torch.cat([pe0[:, :1].float(), g.permute(0, 2, 3, 1).reshape(1, -1, C)], 1)
        m.backbone.pos_embed = torch.nn.Parameter(sp["pe"][grid], requires_grad=False)
        m.backbone.grid = grid

def _grp(s):
    # Nhóm ảnh = thư mục đầu của đường dẫn tương đối (hoặc thư mục cha) -> biết thứ tự các dataset trong test
    for key in ("image", "image_path"):
        parts = str(s.get(key) or "").replace("\\", "/").strip("/").split("/")
        if len(parts) > 1: return parts[0] if key == "image" else parts[-2]
    return "?"

def _guard_step(samples, tail=None):
    g = _S["guard"]; now = time.time(); n = len(samples)
    if g["last_t"] is None:
        s0 = samples[0]
        _log(f"first sample: keys {sorted(s0.keys())} | image {str(s0.get('image'))[:120]!r} | image_path {str(s0.get('image_path'))[:160]!r}")
    elif g["last_n"]:
        g["calls"] += 1
        if g["calls"] > 1 and (g["split_at"] is None or g["last_valid"]):   # bỏ batch đầu (khởi động); tốc độ quy về chi phí mức 0, không gồm model phụ
            per = (now - g["last_t"]) / g["last_n"]
            g["rates"] = (g["rates"] + [max(per - g["last_o"] - g["last_s"], 1e-6) / g["costs"][g["last_k"]]])[-5:]
            if g["last_o"] > 0: g["opt_rates"] = (g["opt_rates"] + [g["last_o"]])[-5:]
            if g["last_s"] > 0: g["sh_rates"] = (g["sh_rates"] + [g["last_s"]])[-5:]
            if g["last_h"] > 0: g["ho_rates"] = (g["ho_rates"] + [g["last_h"] / g["costs"][g["last_k"]]])[-5:]   # v17x: chi phí model head_only quy về mức 0
    if g["shadow_on"] and g["rates"] and g["sh_rates"]:      # lượt "bóng" (chỉ để đo) tắt trước mọi thứ khác nếu dự báo > 80 % ngân sách
        r_ = float(np.median(g["rates"])) * g["costs"][g["k"]] + (float(np.median(g["opt_rates"])) if (g["has_opt"] and g["opt_on"] and g["opt_rates"]) else 0.0)
        rs_, el_, left_ = float(np.median(g["sh_rates"])), now - T0, max(g["n_expected"] - g["n_done"], n)
        if el_ + (r_ + rs_) * left_ > 0.8 * g["budget_s"]:
            g["shadow_on"] = False
            _log(f"shadow OFF after {g['n_done']} images | main {1000 * r_:.2f} + shadow {1000 * rs_:.2f} ms/img | projected {el_ + (r_ + rs_) * left_:.0f}s")
    if g["has_opt"] and g["split_at"] is not None:          # run_split: model phụ chỉ ở đuôi (vị trí >= split_at), bỏ chỉ khi dự báo > hard_s
        on = bool(tail) and g["tail_ok"]
        if on and os.environ.get("UPAR_DROP_OPT"): on = False
        elif on and g["rates"] and not os.environ.get("UPAR_GUARD_OFF"):
            kt = max(g["k"], min(int(g.get("tail_tier", 0) or 0), len(g["scales"]) - 1))
            r = float(np.median(g["rates"])) * g["costs"][kt]
            ro = float(np.median(g["opt_rates"])) if g["opt_rates"] else float(g.get("opt_prior_s", 0.010))   # chưa đo -> giá trị mặc định
            rh = min(float(np.median(g["ho_rates"])) * g["costs"][kt], 0.9 * r) if g["ho_rates"] else 0.0
            el, left = now - T0, max(g["n_expected"] - g["n_done"], n)
            if el + (r - rh + ro) * left > g.get("hard_s", g["budget_s"]):
                g["tail_ok"] = on = False                   # bỏ hẳn cho phần còn lại: đuôi dùng bộ đầu + thresholds_solo
                _log(f"tail optional models DROPPED after {g['n_done']} images | main {1000 * r:.1f} + optional {1000 * ro:.1f}"
                     f"{'' if g['opt_rates'] else ' (prior)'} ms/img | projected {el + (r - rh + ro) * left:.0f}s > hard {g.get('hard_s', g['budget_s']):.0f}s")
        if on != g["opt_on"]:
            _log(f"optional models {'ON' if on else 'OFF'} after {g['n_done']} images ({'tail' if tail else 'head'} part)")
        g["opt_on"] = on
    elif g["has_opt"]:                                      # bỏ / bật lại model phụ (bật lại cần dư 2 %)
        # Khi tắt, mỗi ảnh chỉ chạy model chính làm dự báo "chạy cả 2 model cho phần còn lại" giảm dần -> model phụ được bật lại cho
        # đoạn cuối, dùng hết phần thời gian dư (07/10: hệ số 0.9 cũ quá chặt, server dư ~360 s mà model phụ không được bật lại).
        on = g["opt_on"]
        if os.environ.get("UPAR_GUARD_OFF"): on = True
        elif os.environ.get("UPAR_DROP_OPT"): on = False
        elif g["rates"] and g["opt_rates"]:
            r, ro = float(np.median(g["rates"])) * g["costs"][g["k"]], float(np.median(g["opt_rates"]))
            rh = min(float(np.median(g["ho_rates"])) * g["costs"][g["k"]], 0.9 * r) if g["ho_rates"] else 0.0   # v17x: đuôi không chạy model head_only
            el, left = now - T0, max(g["n_expected"] - g["n_done"], n)
            if left <= g.get("tail_n", 0):                 # v17d (13/10): đuôi tất định — bật model phụ cho tail_n ảnh cuối bất kể trước đó, chỉ cắt ở hard_s
                on = el + (r - rh + ro) * left <= g.get("hard_s", g["budget_s"])
            else:                                          # trước đuôi: chỉ được tắt, không bật lại (độ phủ không phụ thuộc tốc độ máy)
                on = g["opt_on"] and el + (r - rh + ro) * left <= g["budget_s"]
            if on != g["opt_on"]:
                _log(f"optional models {'ON' if on else 'OFF'} after {g['n_done']} images | main {1000 * r:.0f} + optional {1000 * ro:.0f} ms/img"
                     f" | projected {el + (r - rh + ro) * left:.0f}s with, {el + r * left:.0f}s without" + (f" (head_only {1000 * rh:.0f} ms/img not in tail)" if rh else ""))
        g["opt_on"] = on
    k = g["k"]
    if os.environ.get("UPAR_FORCE_TIER"): k = min(int(os.environ["UPAR_FORCE_TIER"]), len(g["scales"]) - 1)
    elif g["rates"] and len(g["scales"]) > 1 and not (g["has_opt"] and g["opt_on"]):
        r, el, left = float(np.median(g["rates"])), now - T0, max(g["n_expected"] - g["n_done"], n)
        k = len(g["scales"]) - 1
        for j in range(len(g["scales"])):                   # mức chính xác nhất còn kịp; quay lên mức tốt hơn cần dư 5 %
            if el + r * g["costs"][j] * left <= g["budget_s"] * (0.95 if j < g["k"] else 1.0): k = j; break
    if g["split_at"] is not None:                           # run_split: mức tail_tier chỉ áp cho đuôi có model phụ
        if tail and g["opt_on"] and g.get("tail_tier"): k = max(k, min(int(g["tail_tier"]), len(g["scales"]) - 1))
    elif g.get("tail_tier") and max(g["n_expected"] - g["n_done"], n) <= g.get("tail_n", 0):   # v17e: đuôi tất định cả về độ phân giải
        k = max(k, min(int(g["tail_tier"]), len(g["scales"]) - 1))
    if k != g["k"]:
        h, w = _size(_S["models"][0][1], k)
        _log(f"tier {g['k']} -> {k} ({h}x{w}) after {g['n_done']} images | "
             f"{1000 * float(np.median(g['rates'])) if g['rates'] else 0:.0f} ms/img at tier-0 cost")
        _set_tier(k); g["k"] = k
    g["last_t"], g["last_n"], g["last_k"] = now, n, k
    return k

def _guard_done(samples, k, tail=None):
    g = _S["guard"]; c = g["groups"]
    pc = g["part_groups"].setdefault(("tail" if g["opt_on"] else "tail-fallback") if tail else "head", {}) if g["split_at"] is not None else None
    for i, s in enumerate(samples):
        grp = _grp(s)
        if grp not in c:
            c[grp] = 0
            if len(c) <= 30: _log(f"new group {grp[:60]!r} at image #{g['n_done'] + i}")
        c[grp] += 1
        if pc is not None: pc[grp] = pc.get(grp, 0) + 1
        ix = s.get("index")
        if isinstance(ix, (int, np.integer)) and int(ix) != g["n_done"] + i:
            g["idx_mismatch"] += 1
            if g["idx_mismatch"] == 1: _log(f"note: sample index {int(ix)} != stream position {g['n_done'] + i} (split uses position)")
    g["n_done"] += len(samples); g["n_tier"][k] += len(samples)
    if g["n_done"] >= g["next_log"] or g["n_done"] == int(g["n_expected"]):
        g["next_log"] = (g["n_done"] // 2880 + 1) * 2880
        el = time.time() - T0
        top = sorted(c.items(), key=lambda kv: -kv[1])[:8]
        n_ = max(g["n_done"], 1)
        tm = (f" | io {1000 * g['t_io'] / n_:.2f} prep {1000 * g['t_prep'] / n_:.2f} model {1000 * g['t_fwd'] / n_:.2f}"
              + (f" shadow {1000 * g['t_shadow'] / max(g['n_shadow'], 1):.2f} ({g['n_shadow']} img)" if g["n_shadow"] else "") + " ms/img")
        _log(f"{g['n_done']} images | {1000 * el / g['n_done']:.1f} ms/img avg{tm} | tier {k} | images per tier {g['n_tier']}"
             + (f" | with optional {g['n_opt']}" if g["has_opt"] else "") + f" | {len(c)} groups, top {top}"
             + (f" | parts {g['part_groups']} | index!=pos {g['idx_mismatch']}" if g["split_at"] is not None else ""))

@torch.no_grad()
def predict_batch(samples):
    if not _S: load_model()
    g = _S["guard"]
    if g["split_at"] is None or not samples: return _predict(samples, None)
    n0 = g["n_done"]; tl = [n0 + i >= g["split_at"] for i in range(len(samples))]   # vị trí trong luồng test, đếm như log "new group"
    if any(tl) and not g["tail_seen"]:
        g["tail_seen"] = True; i = tl.index(True); s = samples[i]
        _log(f"tail starts at image #{n0 + i} | index {s.get('index')!r} | image {str(s.get('image'))[:120]!r} | group {_grp(s)!r}")
    if all(tl) or not any(tl): return _predict(samples, tl[0])
    ih, it = [i for i, t in enumerate(tl) if not t], [i for i, t in enumerate(tl) if t]   # lô vắt qua mốc: đầu trước, đuôi sau
    out = [None] * len(samples)
    for idx, flag in ((ih, False), (it, True)):
        for i, p in zip(idx, _predict([samples[i] for i in idx], flag)): out[i] = p
    return out

def _predict(samples, tail):
    cfg, dev = _S["cfg"], _S["dev"]
    try: k = _guard_step(samples, tail)
    except Exception as e: k = _S["guard"]["k"]; _log(f"guard error {e!r}")
    if "pool" not in _S:
        from concurrent.futures import ThreadPoolExecutor
        _S["pool"] = ThreadPoolExecutor(max_workers=min(16, os.cpu_count() or 1))
    g = _S["guard"]; sync = torch.cuda.synchronize if dev.type == "cuda" else (lambda: None)
    use_opt = g["opt_on"] or not g["has_opt"]
    act = [(m, sp) for m, sp in _S["models"] if (not sp.get("head_only") if use_opt else not sp.get("optional"))]   # v17x: head_only chỉ chạy khi model phụ tắt
    flip = cfg.get("tta_flip", False) or bool(tail and g["has_opt"] and use_opt and cfg.get("tail_flip", False))   # 18/10: flip chỉ ở đuôi có model phụ
    sizes = sorted({(*_size(sp, k), sp["_crop"]) for _, sp in act})
    t0_ = time.time()
    arrs = list(_S["pool"].map(lambda s: _load_resize(s["image_path"], sizes), samples))
    g["t_io"] += time.time() - t0_
    t1 = time.time(); xs = {}
    bad = [i for i, d in enumerate(arrs) if d is None]
    for hw in sizes:                                          # mỗi kích thước: 1 lần stack + đưa lên GPU, dùng chung cho các model cùng kích thước
        a = np.stack([d[hw] if d is not None else np.zeros((hw[0], hw[1], 3), np.uint8) for d in arrs])     # hw = (h, w, crop)
        xs[hw] = torch.from_numpy(a).to(dev).permute(0, 3, 1, 2).float().div_(255)
    g["t_prep"] += time.time() - t1
    z, wsum, t_opt, x0, t_ho = None, 0.0, 0.0, None, 0.0
    for m, sp in act:
        t1 = time.time(); h, w = _size(sp, k)
        x = xs[(h, w, sp["_crop"])]
        if sp.get("wb", False): x = _grayworld_b(x)
        x = (x - sp["_mean"]) / sp["_std"]
        if bad: x[bad] = 0.0                                  # ảnh đọc lỗi: tensor 0 như _prep() cũ
        t2 = time.time()
        with torch.autocast(dev.type, enabled=dev.type == "cuda"):
            zi = m(x).float()
            if flip:
                zi = (zi + m(torch.flip(x, dims=[3])).float()) / 2
        sync(); g["t_fwd"] += time.time() - t2
        if sp.get("head_only"): t_ho += time.time() - t2
        if x0 is None: x0 = x
        z = sp["_w"] * zi if z is None else z + sp["_w"] * zi; wsum = wsum + sp["_w"]
        if sp.get("optional"): t_opt += time.time() - t1
    g["last_o"] = t_opt / max(len(samples), 1); g["last_h"] = t_ho / max(len(samples), 1)
    if g["split_at"] is not None:                           # lô nhỏ (vắt mốc) và lô đủ cỡ đầu tiên có model phụ (chạy nguội): không dùng để đo tốc độ
        full = len(samples) >= int(g.get("opt_min_n", 16)); opt_run = use_opt and g["has_opt"]
        if opt_run and full: g["opt_full"] += 1
        g["last_valid"] = full and (not opt_run or g["opt_full"] > 1)
    n_sh, t_sh = int(cfg.get("shadow_passes", 0) or 0), 0.0
    if n_sh and g["shadow_on"] and x0 is not None:          # chỉ để đo chi phí thêm 1 model: chạy lại model đầu trên ảnh lật, bỏ kết quả
        t3 = time.time()
        with torch.autocast(dev.type, enabled=dev.type == "cuda"):
            for _ in range(n_sh): _S["models"][0][0](torch.flip(x0, dims=[3]))
        sync(); t_sh = time.time() - t3; g["t_shadow"] += t_sh; g["n_shadow"] += len(samples)
    g["last_s"] = t_sh / max(len(samples), 1)
    if use_opt:
        off = _S["off"]; g["n_opt"] += len(samples) if g["has_opt"] else 0
    else:                                                   # chỉ model chính: logit chuẩn hoá lại theo tổng trọng số, ngưỡng riêng
        z, off = z / wsum, _S["off_solo"]
    p = torch.sigmoid(z - off)
    p = torch.nan_to_num(p, nan=0.0).clamp(0.0, 1.0).cpu().numpy()
    names = samples[0].get("attribute_names") or _S["names"]
    if list(names) != list(_S["names"]):                   # trả đúng thứ tự BTC yêu cầu
        p = p[:, [_S["names"].index(n) for n in names]]
    try: _guard_done(samples, k, tail)
    except Exception as e: _log(f"guard error {e!r}")
    return p.tolist()

def predict_image(sample):
    return predict_batch([sample])[0]

### 11c. Ngưỡng và `config.json`

- `m0`, `m1`: logit val của F8c / L2c tính bằng chính `run.py`, tức weights w8 ở 224x112, cùng tiền xử lý với server, không lật.
- `G`, `R`: logit val 224x112 của g11 / RADIO, lưu lúc train.
- `thresholds_solo` (đầu) = hiệu chỉnh trên 0.6·m0 + 0.4·m1.
- `thresholds` (đuôi) = hiệu chỉnh trên 0.24·m0 + 0.16·m1 + 0.3·G + 0.3·R.
- Cả hai đều in-sample trên val resplit.
- `config.json` có cùng các trường và cùng thứ tự với bản gốc.

In [ ]:
print("[cell] thresholds + config", flush=True)
META_YAML = '# Code submissions are executed by the challenges ingestion program, which\n# imports run.py and calls predict_image() / predict_batch(). Nothing in this\n# file is used for inference, it exists only so the platform never treats the\n# archive as a result submission. Leave it as-is.\ncommand: python3 -c "print(\'Submission package ready. The ingestion program imports run.py\')"\n'
TEXT_CRC = {"run.py": 0x9C6C1C26, "model.py": 0xB29B31A6, "metadata.yaml": 0xD009A272}   # CRC 3 file văn bản trong zip bản gốc

def package_texts():
    """Nội dung run.py / model.py / metadata.yaml của gói nộp, lấy từ các cell %%module (không ghi ra đĩa, chỉ đưa vào zip)."""
    t = {"run.py": (SRC["run"].strip("\n").replace("\n", "\r\n") + "\r\n").encode("utf-8"),     # bản gốc: CRLF
         "model.py": ("\n" + SRC["model"].strip("\n") + "\n").encode("utf-8"),                   # bản gốc: LF, mở đầu bằng 1 dòng trống
         "metadata.yaml": META_YAML.encode("utf-8")}
    for k, b in t.items():
        assert zlib.crc32(b) == TEXT_CRC[k], f"{k}: CRC {zlib.crc32(b):#010x} khác bản gốc {TEXT_CRC[k]:#010x} (cell bị sửa?)"
    return t

def make_config(thr_tail, thr_solo):
    return {"models": MODELS, "tta_flip": False, "attributes": ATTR,
            "thresholds": thr_tail, "thresholds_calibrated": thr_tail, "thr_shrink": 1.0, "thresholds_solo": thr_solo,
            "cpu_int8": False, "int8_skip": [], "smoothquant_alpha": None,
            "version": f"{ZIP_NAME}: head(<{SPLIT_AT}) F8c + L2c thr=solo cal 0.6 m0 + 0.4 m1 | tail(>={SPLIT_AT}) F8c 0.24 + L2c 0.16 + G 0.3 + R 0.3 tier 1 + flip | run_split | e2e notebook",
            "time_guard": {"budget_s": 266.0, "hard_s": 280.0, "n_expected": 29248, "tail_n": 0, "tail_tier": 1, "opt_prior_s": 0.01},
            "val_rows": len(VA),
            "calibrated_on": "val resplit 0710 in-sample; đầu = logit run.py (0.6 m0 + 0.4 m1), đuôi = 0.24 m0 + 0.16 m1 (run.py) + 0.3 G + 0.3 R (logit 224x112 lúc train)",
            "split": {"at": SPLIT_AT, "by": "position", "note": "test: MEVID #0-28094, Synth #28095-29247"},
            "tail_flip": True}

def write_config(cfg):
    (PKG / "assets/config.json").write_text(json.dumps(cfg, indent=1, ensure_ascii=False), encoding="utf-8")

def run_val_logits(R, frame, n_models=2, bs=64):
    """Logit val của n model đầu trong gói, đi đúng đường tiền xử lý của run.py (PIL resize -> tensor -> chuẩn hoá trên GPU)."""
    from concurrent.futures import ThreadPoolExecutor
    paths = [str(DATA / p) for p in frame.image]; dv = R._S["dev"]; outs = []
    with ThreadPoolExecutor(max_workers=min(16, os.cpu_count() or 1)) as tp:
        for m, sp in R._S["models"][:n_models]:
            key = (*R._size(sp, 0), sp["_crop"]); zz = []
            for b0 in range(0, len(paths), bs):
                arr = np.stack([d[key] if d is not None else np.zeros((key[0], key[1], 3), np.uint8)
                                for d in tp.map(lambda p: R._load_resize(p, [key]), paths[b0:b0 + bs])])
                x = (torch.from_numpy(arr).to(dv).permute(0, 3, 1, 2).float().div_(255) - sp["_mean"]) / sp["_std"]
                with torch.no_grad(), torch.autocast(dv.type, enabled=dv.type == "cuda"): zz.append(m(x).float().cpu().numpy())
                if (b0 // bs) % 50 == 0: print(f"  {sp['arch']}: {b0}/{len(paths)}", flush=True)
            outs.append(np.concatenate(zz).astype(np.float64))
    return outs

if DO_PACK:
    package_texts()
    write_config(make_config([0.5] * 40, [0.5] * 40))           # config tạm để run.py nạp được model
    os.environ["UPAR_GUARD_OFF"] = "1"
    R = load_source("run", PKG / "run.py"); R.load_model()
    m0, m1 = run_val_logits(R, VA)
    os.environ.pop("UPAR_GUARD_OFF", None); R._S.clear(); del R; gc.collect(); torch.cuda.empty_cache()
    np.save(WORK / "val_logit_m0.npy", m0.astype(np.float32)); np.save(WORK / "val_logit_m1.npy", m1.astype(np.float32))
    G = np.load(run_file("e2e_g11", "val_logits_224x112.npy")).astype(np.float64)
    RD = np.load(run_file("e2e_radio", "val_logits_224x112.npy")).astype(np.float64)
    assert m0.shape == m1.shape == G.shape == RD.shape == (len(VA), 40)
    THR_SOLO, s_h = calibrate(VA, sigm(0.6 * m0 + 0.4 * m1))
    THR_TAIL, s_t = calibrate(VA, sigm(0.24 * m0 + 0.16 * m1 + 0.3 * G + 0.3 * RD))
    THR_SOLO, THR_TAIL = [float(x) for x in THR_SOLO], [float(x) for x in THR_TAIL]
    print(f"val in-sample: đầu (F8c + L2c) {s_h:.4f} | đuôi (4 model) {s_t:.4f}")
    print(evaluate(VA, sigm(0.6 * m0 + 0.4 * m1 - logit(np.array(THR_SOLO)))).round(4).to_string())
    CFG_FINAL = make_config(THR_TAIL, THR_SOLO); write_config(CFG_FINAL)

### 11d. Zip

Cùng thứ tự mục và cách nén như bản gốc:
- `run.py`, `model.py`, `metadata.yaml`, `assets/config.json`: deflate mức 1.
- `weights_*.pt`: lưu không nén (STORED).

In [ ]:
print("[cell] zip", flush=True)
if DO_PACK:
    ZO = WORK / ZIP_NAME; TEXTS = package_texts()
    with zipfile.ZipFile(ZO, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as zo:
        for k in ("run.py", "model.py", "metadata.yaml"): zo.writestr(k, TEXTS[k])
        zo.writestr("assets/config.json", json.dumps(CFG_FINAL, indent=1, ensure_ascii=False))
        for i in range(len(PACK)):
            zi = zipfile.ZipInfo(f"assets/weights_{i}.pt", date_time=(2026, 10, 19, 0, 0, 0)); zi.compress_type = zipfile.ZIP_STORED
            with open(PKG / "assets" / f"weights_{i}.pt", "rb") as fi, zo.open(zi, "w", force_zip64=True) as fo: shutil.copyfileobj(fi, fo, 1 << 24)
    with zipfile.ZipFile(ZO) as z:
        assert z.testzip() is None
        for x in z.infolist(): print(f"  {x.filename:22s} {x.file_size:>11d} B  CRC {x.CRC:#010x}")
    h = hashlib.md5()
    with open(ZO, "rb") as f:
        while (b := f.read(1 << 24)): h.update(b)
    ZIP_MD5 = h.hexdigest()
    print(f"ĐÃ GHI {ZO} ({ZO.stat().st_size / 1e9:.2f} GB) md5 {ZIP_MD5} (bản gốc e9bb5cdf...; weights train lại nên md5 khác)")

### 11e. Tự kiểm gói nộp

- Giải nén `assets/` từ zip.
- Nạp `run.py` (mã của cell `%%module run`, `__file__` trỏ vào thư mục giải nén) và chạy `predict_batch` đúng như chương trình chấm, trên 256 ảnh val.
- **Đầu**: so với sigmoid(0.6·m0 + 0.4·m1 − logit(ngưỡng solo)). Phải lệch |Δp| < 0.05 và trùng dấu > 99.5 %.
- **Đuôi** (`UPAR_SPLIT_AT=0`, 4 model + lật ngang): so với logit 224x112 không lật. Phải trùng dấu > 95 %.

In [ ]:
print("[cell] self-check", flush=True)
if DO_PACK:
    CK = Path("/tmp/upar_zipcheck") if IS_KAGGLE else WORK / "zipcheck"; shutil.rmtree(CK, ignore_errors=True)
    with zipfile.ZipFile(ZO) as z:
        for k, b in package_texts().items(): assert z.read(k) == b, f"{k} trong zip khác mã của cell"
        for n in z.namelist():
            if n.startswith("assets/"): z.extract(n, CK)
    rows = VA.image.tolist(); paths = [str(DATA / r) for r in rows]; idx = np.random.RandomState(1).choice(len(rows), 256, replace=False)
    def check(tag, z_ref, thr):
        R = load_source("run", CK / "run.py"); R.load_model(); t1 = time.time()
        smp = [dict(attribute_names=CFG_FINAL["attributes"], image=rows[i], image_path=paths[i], index=int(i)) for i in idx]
        p = np.array(R.predict_batch(smp[:128]) + R.predict_batch(smp[128:]))
        tt = time.time() - t1; thr_ = np.clip(np.array(thr), 1e-4, 1 - 1e-4); q = sigm(z_ref[idx] - np.log(thr_ / (1 - thr_)))
        dp = np.abs(p - q); agree = float(((p > .5) == (q > .5)).mean())
        print(f"tự kiểm {tag}: |Δp| TB {dp.mean():.5f} max {dp.max():.4f}, trùng dấu {agree:.4f} | {1000 * tt / 256:.1f} ms/ảnh (gồm khởi động)", flush=True)
        R._S.clear(); del R; gc.collect(); torch.cuda.empty_cache()
        return dp, agree
    os.environ.pop("UPAR_GUARD_OFF", None)                     # chốt chặn thời gian bật như trên server
    dp, ag = check("ĐẦU (F8c + L2c, ngưỡng solo)", 0.6 * m0 + 0.4 * m1, THR_SOLO)
    assert dp.max() < 0.05 and ag > 0.995, "đầu lệch logit val"
    os.environ["UPAR_SPLIT_AT"] = "0"                          # coi mọi ảnh là đuôi
    dpt, agt = check("ĐUÔI (4 model + lật, so với logit 224 không lật)", 0.24 * m0 + 0.16 * m1 + 0.3 * G + 0.3 * RD, THR_TAIL)
    os.environ.pop("UPAR_SPLIT_AT", None)
    shutil.rmtree(CK, ignore_errors=True)
    assert agt > 0.95, "đuôi lệch quá xa"
    print("TỰ KIỂM ĐẠT")

In [ ]:
print("[cell] compare original", flush=True)
# Tuỳ chọn: gắn zip bản gốc (SUBMIT_split_x2f_cmask_f8cL2c_tail4flip_gpu_w8.zip) làm Input để so cấu trúc và ngưỡng.
orig = next((d / ZIP_NAME for d in input_dirs() if (d / ZIP_NAME).exists()), None)
if DO_PACK and orig is not None:
    with zipfile.ZipFile(orig) as zA, zipfile.ZipFile(ZO) as zB:
        for x in zA.infolist():
            y = zB.getinfo(x.filename)
            print(f"  {x.filename:22s} gốc {x.file_size:>11d} {x.CRC:#010x} | mới {y.file_size:>11d} {y.CRC:#010x} {'TRÙNG' if x.CRC == y.CRC else 'khác'}")
        cA, cB = json.loads(zA.read("assets/config.json")), json.loads(zB.read("assets/config.json"))
    assert list(cA) == list(cB), "khoá config khác bản gốc"
    strip = lambda m: {k: v for k, v in m.items() if k != "source"}
    assert [strip(m) for m in cA["models"]] == [strip(m) for m in cB["models"]], "spec model khác bản gốc"
    for k in ("thresholds", "thresholds_solo"):
        a_, b_ = np.array(cA[k]), np.array(cB[k]); print(f"  {k}: {int((np.abs(a_ - b_) > 1e-9).sum())}/40 khác, max |d| {np.abs(a_ - b_).max():.3f}")
    print("cấu trúc config + spec model trùng bản gốc; weights / ngưỡng khác do train lại")
else:
    print("bỏ qua (không có zip bản gốc trong Input hoặc không chạy pack)")
print(f"Xong sau {(time.time() - T_START) / 3600:.2f} h | dừng giữa chừng: {sorted(PAUSED) or 'không'}")